In [ ]:
pip install optuna openpyxl


In [ ]:
pip install plotly kaleido


In [ ]:
# 노트북 셀에 입력한 뒤 실행
!pip install cma


In [ ]:
# 1) 간단하게 pip 매직
!pip install cmaes


In [ ]:
 #Parameter Optimization 1 (아무 생각 없이 normalize 한 결과)

In [ ]:
# import os
# import pandas as pd
# import numpy as np
# import optuna
# from sklearn.preprocessing import MinMaxScaler

# # 1) 경로 설정
# # input_csv     = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# # results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"

# input_csv     = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# results_folder = r"C:\Users\seung\OneDrive\주식\Results"

# os.makedirs(results_folder, exist_ok=True)

# # 2) 데이터 로딩 및 기간 필터링
# df = pd.read_csv(input_csv, encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# # 3) 최적화용 입력 변수(지표) 리스트
# indicator_cols = [
#     '거래량', '변동 %', 'RSI (14일)',
#     '볼린저밴드 상단', '볼린저밴드 하단',
#     'MACD', 'MACD 시그널',
#     'SMA 5일', 'SMA 10일', 'SMA 20일',
#     'SMA 60일', 'SMA 120일', 'SMA 200일',
#     '가격 상승률 (2주)', '거래량 상승률 (2주)',
#     '가격 상승률 (3개월)', '거래량 상승률 (3개월)',
#     '가격 상승률 (6개월)', '거래량 상승률 (6개월)',
#     '가격 상승률 (1년)', '거래량 상승률 (1년)'
# ]

# # 4) Min–Max 스케일링 (0~1)
# scaler = MinMaxScaler()
# # 거래량은 로그 변환 후 스케일링
# df['거래량_log'] = np.log1p(df['거래량'])
# scale_cols = ['거래량_log'] + [c for c in indicator_cols if c != '거래량']
# df_norm = df.copy()
# df_norm[[*scale_cols]] = scaler.fit_transform(df[scale_cols])

# # 5) 백테스트 함수 정의
# def backtest(weights, threshold):
#     cash, shares = 10_000.0, 0.0
#     for idx, row in df_norm.iterrows():
#         # score 계산 (거래량_log_norm 포함, 거래량 원본 컬럼은 제외)
#         score = sum(
#             w * row[col] 
#             for w, col in zip(weights, scale_cols)
#         )
#         price = df.loc[idx, '종가']

#         # 매수: 현금만 있고 score > threshold
#         if shares == 0 and score > threshold:
#             shares = cash / price
#             cash = 0.0
#         # 매도: 주식만 있고 score < threshold
#         elif shares > 0 and score < threshold:
#             cash = shares * price
#             shares = 0.0

#     # 기간 종료 시 전량 청산
#     final_value = cash + shares * df.iloc[-1]['종가']
#     roi = (final_value) / 10_000.0 * 100
#     return roi

# # 6) Optuna 목적 함수
# def objective(trial):
#     # (scale_cols 개수)만큼 가중치 제안
#     weights = [
#         trial.suggest_uniform(f"w_{i}", -2.0, 2.0)
#         for i in range(len(scale_cols))
#     ]
#     # threshold 제안: [0, 2*N] 구간
#     thr = trial.suggest_uniform("threshold", 0.0, 2.0 * len(scale_cols))
#     # ROI 최대화
#     return backtest(weights, thr)

# # 7) 최적화 실행
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=100)

# # 8) 모든 트라이얼 파라미터 + ROI → DataFrame → Excel 저장
# trials_df = study.trials_dataframe().rename(columns={
#     'number': 'trial_number',
#     'value':  'ROI'
# })
# # params_* 컬럼만 뽑아서 재정렬
# cols = ['trial_number', 'ROI'] + [c for c in trials_df.columns if c.startswith('params_')]
# trials_df[cols].to_excel(
#     os.path.join(results_folder, "optimization_trials.xlsx"),
#     index=False
# )

# # 9) 최적 파라미터 & ROI 출력
# print("⭐️ Best ROI (%):", round(study.best_value, 2))
# print("⭐️ Best Params:")
# for k, v in study.best_params.items():
#     print(f"   {k}: {v:.4f}")

# # 10) 최적 파라미터로 다시 백테스트 (검증)
# best_weights = [study.best_params[f"w_{i}"] for i in range(len(scale_cols))]
# best_thr     = study.best_params["threshold"]
# best_roi     = backtest(best_weights, best_thr)
# print(f"▶ Validation ROI with best params: {best_roi:.2f}%")


In [ ]:
 #Parameter Optimization 1-2 
#     각각의 StrategyCondition 서브클래스로 매수·매도 시그널을 판별할 조건을 정의합니다:

# 클래스	매수/매도	설명
# RSIBelow(th)	매수	RSI(14일) < th
# BollingerNearLower(buf)	매수	종가 < Lower BB × (1 + buf) (하단 밴드 근처)
# MACDPositive	매수	MACD > MACD 시그널
# MA5AboveMA10	매수	5일 이동평균 > 10일 이동평균
# MA5BelowMA60	매수	5일 이동평균 < 60일 이동평균
# TwoWeekPriceLow(pct)	매수	2주간 누적 상승률 < pct (%)
# RSISell(th)	매도	RSI(14일) > th


In [ ]:
# import os
# import pandas as pd
# import numpy as np
# import optuna
# from sklearn.preprocessing import MinMaxScaler

# # 1) 경로 설정
# input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
# os.makedirs(results_folder, exist_ok=True)

# # 2) 데이터 로딩 및 기간 필터링
# df = pd.read_csv(input_csv, encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# # 3) 거래량 로그 정규화
# df['거래량_log'] = np.log1p(df['거래량'])
# # 2주·3개월 가격/거래량 상승률 raw 컬럼은 그대로 사용

# # 4) 전략 조건 클래스 정의
# class StrategyCondition:
#     def check(self, row): raise NotImplementedError()

# class RSIBelow(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] < self.th

# class BollingerNearLower(StrategyCondition):
#     def __init__(self, buf): self.buf = buf
#     def check(self, row):
#         return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)

# class MACDPositive(StrategyCondition):
#     def check(self, row): return row['MACD'] > row['MACD 시그널']

# class MA5AboveMA10(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] > row['SMA 10일']

# class MA5BelowMA60(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] < row['SMA 60일']

# class TwoWeekPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (2주)'] < self.pct

# class ThreeMonthPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (3개월)'] < self.pct

# class TwoWeekVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (2주)'] < self.pct

# class ThreeMonthVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct

# class RSISell(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] > self.th

# # 5) 백테스트 함수
# def backtest(rsi_buy_th, boll_buffer,
#              tw_price_th, tm_price_th,
#              tw_vol_th, tm_vol_th,
#              rsi_sell_th):
#     cash, shares = 10_000.0, 0.0

#     # 복합 전략 인스턴스
#     buy_conds = [
#         RSIBelow(rsi_buy_th),
#         BollingerNearLower(boll_buffer),
#         MACDPositive(),
#         MA5AboveMA10(),
#         MA5BelowMA60(),
#         TwoWeekPriceLow(tw_price_th),
#         ThreeMonthPriceLow(tm_price_th),
#         TwoWeekVolumeLow(tw_vol_th),
#         ThreeMonthVolumeLow(tm_vol_th),
#     ]
#     sell_conds = [RSISell(rsi_sell_th)]
    
#     for _, row in df.iterrows():
#         price = row['종가']

#         # 매수
#         if shares == 0 and all(cond.check(row) for cond in buy_conds):
#             shares = cash / price
#             cash = 0.0

#         # 매도
#         elif shares > 0 and any(cond.check(row) for cond in sell_conds):
#             cash = shares * price
#             shares = 0.0

#     # 최종 청산
#     final_val = cash + shares * df.iloc[-1]['종가']
#     return (final_val - 10_000.0) / 10_000.0 * 100  # ROI (%)

# # 6) Optuna 목적 함수
# def objective(trial):
#     rsi_buy_th    = trial.suggest_uniform("rsi_buy_th",    0.0, 100.0)
#     boll_buffer   = trial.suggest_uniform("boll_buffer",   0.0,   0.1)
#     tw_price_th   = trial.suggest_uniform("tw_price_th",   0.0,  20.0)
#     tm_price_th   = trial.suggest_uniform("tm_price_th",   0.0,  50.0)
#     tw_vol_th     = trial.suggest_uniform("tw_vol_th",     0.0, 100.0)
#     tm_vol_th     = trial.suggest_uniform("tm_vol_th",     0.0, 300.0)
#     rsi_sell_th   = trial.suggest_uniform("rsi_sell_th",   0.0, 100.0)

#     return backtest(
#         rsi_buy_th, boll_buffer,
#         tw_price_th, tm_price_th,
#         tw_vol_th, tm_vol_th,
#         rsi_sell_th
#     )

# # 7) 최적화 실행
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=100)

# # 8) 결과 저장
# trials_df = study.trials_dataframe().rename(columns={
#     'number': 'trial_number',
#     'value':  'ROI'
# })
# cols = ['trial_number', 'ROI'] + [c for c in trials_df.columns if c.startswith('params_')]
# trials_df[cols].to_excel(
#     os.path.join(results_folder, "rule_based_optuna_trials.xlsx"),
#     index=False
# )

# # 9) 최적 파라미터 & 검증
# print("⭐️ Best ROI (%):", round(study.best_value, 2))
# print("⭐️ Best Params:")
# for k, v in study.best_params.items():
#     print(f"   {k}: {v:.4f}")

# best = study.best_params
# valid_roi = backtest(
#     best['rsi_buy_th'], best['boll_buffer'],
#     best['tw_price_th'], best['tm_price_th'],
#     best['tw_vol_th'], best['tm_vol_th'],
#     best['rsi_sell_th']
# )
# print(f"▶ Validation ROI with best params: {valid_roi:.2f}%")



In [ ]:
 #Parameter Optimization 1-3
#     각각의 StrategyCondition 서브클래스로 매수·매도 시그널을 판별할 조건을 정의합니다:

# 클래스	매수/매도	설명
# RSIBelow(th)	매수	RSI(14일) < th
# BollingerNearLower(buf)	매수	종가 < Lower BB × (1 + buf) (하단 밴드 근처)
# MACDPositive	매수	MACD > MACD 시그널
# MA5AboveMA10	매수	5일 이동평균 > 10일 이동평균
# MA5BelowMA60	매수	5일 이동평균 < 60일 이동평균
# TwoWeekPriceLow(pct)	매수	2주간 누적 상승률 < pct (%)
# RSISell(th)	매도	RSI(14일) > th

##   여기에다 매크로 vix, high yield spread 반영 (이건 우선 적용)

buy 100% , sell 100%

In [ ]:
#위에다가 몇번의 매도 매수가 있었는지 추가

In [ ]:
# import os
# import pandas as pd
# import numpy as np
# import optuna
# from sklearn.preprocessing import MinMaxScaler

# # ─── 1) 경로 설정 ───────────────────────────────────────────
# # input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# # macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
# # results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"

# input_csv      = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# macro_folder   = r"C:\Users\seung\OneDrive\주식\Macro Data"
# results_folder = r"C:\Users\seung\OneDrive\주식\Results"

# os.makedirs(results_folder, exist_ok=True)

# # ─── 2) 데이터 로딩 및 머지 ─────────────────────────────────
# df = pd.read_csv(input_csv, encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2015-01-02') & (df['날짜'] < '2025-06-24')].reset_index(drop=True)

# vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"), parse_dates=[0], encoding='utf-8-sig')
# vix_df.columns = ['날짜','VIX']
# hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
#                      parse_dates=[0], encoding='utf-8-sig')
# hys_df.columns = ['날짜','HYS']

# df = df.merge(vix_df, on='날짜', how='left').merge(hys_df, on='날짜', how='left')
# df['거래량_log'] = np.log1p(df['거래량'])

# # ─── 3) 전략 조건 클래스 ────────────────────────────────────
# class StrategyCondition:
#     def check(self, row): raise NotImplementedError()
# class RSIBelow(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] < self.th
# class BollingerNearLower(StrategyCondition):
#     def __init__(self, buf): self.buf = buf
#     def check(self, row): return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)
# class MACDPositive(StrategyCondition):
#     def check(self, row): return row['MACD'] > row['MACD 시그널']
# class MA5AboveMA10(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] > row['SMA 10일']
# class MA5BelowMA60(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] < row['SMA 60일']
# class TwoWeekPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (2주)'] < self.pct
# class ThreeMonthPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (3개월)'] < self.pct
# class TwoWeekVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (2주)'] < self.pct
# class ThreeMonthVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct
# class RSISell(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] > self.th

# # ─── 4) 백테스트 함수 (ROI, buy_count, sell_count 반환) ────────────────
# def backtest_count(params):
#     cash, shares = 10_000.0, 0.0
#     total_invested = 10_000.0
#     buy_count = 0
#     sell_count = 0

#     buy_conds = [
#         RSIBelow(params['rsi_buy_th']),
#         BollingerNearLower(params['boll_buffer']),
#         MACDPositive(),
#         MA5AboveMA10(),
#         MA5BelowMA60(),
#         TwoWeekPriceLow(params['tw_price_th']),
#         ThreeMonthPriceLow(params['tm_price_th']),
#         TwoWeekVolumeLow(params['tw_vol_th']),
#         ThreeMonthVolumeLow(params['tm_vol_th']),
#     ]
#     sell_conds = [RSISell(params['rsi_sell_th'])]

#     for _, row in df.iterrows():
#         price, vix = row['종가'], row['VIX']

#         # (1) 매크로 강제 매수
#         if shares == 0 and not pd.isna(vix) and vix >= 60:
#             shares = cash / price
#             cash = 0.0
#             buy_count += 1
#             continue

#         # (2) 룰 기반 매수
#         if shares == 0 and all(cond.check(row) for cond in buy_conds):
#             shares = cash / price
#             cash = 0.0
#             buy_count += 1

#         # (3) 룰 기반 매도
#         elif shares > 0 and any(cond.check(row) for cond in sell_conds):
#             cash = shares * price
#             shares = 0.0
#             sell_count += 1

#     final_val = cash + shares * df.iloc[-1]['종가']
#     roi = (final_val - 10_000.0) / 10_000.0 * 100
#     return roi, buy_count, sell_count

# # ─── 5) Optuna 최적화 ─────────────────────────────────────
# def objective(trial):
#     return backtest_count({
#         'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
#         'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
#         'tw_price_th': trial.suggest_float("tw_price_th", 0, 20),
#         'tm_price_th': trial.suggest_float("tm_price_th", 0, 50),
#         'tw_vol_th':   trial.suggest_float("tw_vol_th",   0, 100),
#         'tm_vol_th':   trial.suggest_float("tm_vol_th",   0, 300),
#         'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
#     })[0]

# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=100)

# # ─── 6) 각 trial별 ROI, buy_count, sell_count 수집 ─────────────
# records = []
# for trial in study.trials:
#     roi, bc, sc = backtest_count(trial.params)
#     rec = {'trial': trial.number, 'ROI': roi, **{f'params_{k}': v for k, v in trial.params.items()}}
#     rec['buy_count']  = bc
#     rec['sell_count'] = sc
#     records.append(rec)

# results_df = pd.DataFrame(records)

# # ─── 7) 결과 저장 & 출력 ────────────────────────────────────
# excel_path = os.path.join(results_folder, "macro_optuna_trials_with_counts.xlsx")
# results_df.to_excel(excel_path, index=False)

# print("⭐️ Best ROI (%):", round(study.best_value, 2))
# print("⭐️ Best Params:", study.best_params)
# print("✅ Trials with buy/sell counts saved to:", excel_path)


In [ ]:
# 다른 형식의 sampling techinique 쓰는거 (두개 TPE + CAE)

In [ ]:
# import os
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import optuna
# from sklearn.preprocessing import MinMaxScaler

# # ────────────────────────────────────────────────────────────────
# # 1) 경로 설정
# # ────────────────────────────────────────────────────────────────
# input_csv      = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# macro_folder   = r"C:\Users\seung\OneDrive\주식\Macro Data"
# results_folder = r"C:\Users\seung\OneDrive\주식\Results"
# os.makedirs(results_folder, exist_ok=True)

# # ────────────────────────────────────────────────────────────────
# # 2) 데이터 로딩 및 전처리
# # ────────────────────────────────────────────────────────────────
# df = pd.read_csv(input_csv, encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2019-01-02') & (df['날짜'] < '2025-06-24')].reset_index(drop=True)

# vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"), parse_dates=[0], encoding='utf-8-sig')
# vix_df.columns = ['날짜','VIX']
# hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
#                      parse_dates=[0], encoding='utf-8-sig')
# hys_df.columns = ['날짜','HYS']

# df = df.merge(vix_df, on='날짜', how='left').merge(hys_df, on='날짜', how='left')
# df['거래량_log'] = np.log1p(df['거래량'])

# # ────────────────────────────────────────────────────────────────
# # 3) 전략 조건 클래스
# # ────────────────────────────────────────────────────────────────
# class StrategyCondition:
#     def check(self, row): raise NotImplementedError()

# class RSIBelow(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] < self.th

# class BollingerNearLower(StrategyCondition):
#     def __init__(self, buf): self.buf = buf
#     def check(self, row):
#         return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)

# class MACDPositive(StrategyCondition):
#     def check(self, row): return row['MACD'] > row['MACD 시그널']

# class MA5AboveMA10(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] > row['SMA 10일']

# class MA5BelowMA60(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] < row['SMA 60일']

# class TwoWeekPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (2주)'] < self.pct

# class ThreeMonthPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (3개월)'] < self.pct

# class TwoWeekVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (2주)'] < self.pct

# class ThreeMonthVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct

# class RSISell(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] > self.th

# # ────────────────────────────────────────────────────────────────
# # 4) 백테스트 함수 (ROI만 반환)
# # ────────────────────────────────────────────────────────────────
# def backtest_roi(params):
#     cash, shares = 10_000.0, 0.0
#     buy_conds = [
#         RSIBelow(params['rsi_buy_th']),
#         BollingerNearLower(params['boll_buffer']),
#         MACDPositive(),
#         MA5AboveMA10(),
#         MA5BelowMA60(),
#         TwoWeekPriceLow(params['tw_price_th']),
#         ThreeMonthPriceLow(params['tm_price_th']),
#         TwoWeekVolumeLow(params['tw_vol_th']),
#         ThreeMonthVolumeLow(params['tm_vol_th']),
#     ]
#     sell_conds = [RSISell(params['rsi_sell_th'])]

#     for _, row in df.iterrows():
#         price, vix = row['종가'], row['VIX']
#         # 매크로 강제 매수
#         if shares == 0 and not pd.isna(vix) and vix >= 60:
#             shares, cash = cash/price, 0.0
#             continue
#         # 룰 기반 매수
#         if shares == 0 and all(cond.check(row) for cond in buy_conds):
#             shares, cash = cash/price, 0.0
#         # 룰 기반 매도
#         elif shares > 0 and any(cond.check(row) for cond in sell_conds):
#             cash, shares = shares*price, 0.0

#     final_val = cash + shares * df.iloc[-1]['종가']
#     return (final_val - 10_000.0) / 10_000.0 * 100

# # ────────────────────────────────────────────────────────────────
# # 5) 1단계: TPE로 빠른 탐색
# # ────────────────────────────────────────────────────────────────
# def objective_tpe(trial):
#     return backtest_roi({
#         'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
#         'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
#         'tw_price_th': trial.suggest_float("tw_price_th", 0, 20),
#         'tm_price_th': trial.suggest_float("tm_price_th", 0, 50),
#         'tw_vol_th':   trial.suggest_float("tw_vol_th",   0, 100),
#         'tm_vol_th':   trial.suggest_float("tm_vol_th",   0, 300),
#         'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
#     })

# tpe_study = optuna.create_study(direction="maximize",
#                                 sampler=optuna.samplers.TPESampler())
# tpe_study.optimize(objective_tpe, n_trials=200)

# best_tpe = tpe_study.best_params
# print("▶ [TPE] Best params:", best_tpe, "ROI:", tpe_study.best_value)

# # ────────────────────────────────────────────────────────────────
# # 6) 2단계: CMA-ES로 정밀 탐색을 위한 파라미터 범위 좁히기
# # ────────────────────────────────────────────────────────────────
# global_bounds = {
#     'rsi_buy_th':  (0,100),
#     'boll_buffer': (0,0.1),
#     'tw_price_th': (0,20),
#     'tm_price_th': (0,50),
#     'tw_vol_th':   (0,100),
#     'tm_vol_th':   (0,300),
#     'rsi_sell_th': (0,100),
# }
# narrow_bounds = {}
# for k,(lo,hi) in global_bounds.items():
#     bp = best_tpe[k]
#     delta = 0.2 * (hi-lo)
#     narrow_bounds[k] = (max(lo, bp-delta), min(hi, bp+delta))

# def objective_cma(trial):
#     return backtest_roi({
#         'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  *narrow_bounds['rsi_buy_th']),
#         'boll_buffer': trial.suggest_float("boll_buffer", *narrow_bounds['boll_buffer']),
#         'tw_price_th': trial.suggest_float("tw_price_th", *narrow_bounds['tw_price_th']),
#         'tm_price_th': trial.suggest_float("tm_price_th", *narrow_bounds['tm_price_th']),
#         'tw_vol_th':   trial.suggest_float("tw_vol_th",   *narrow_bounds['tw_vol_th']),
#         'tm_vol_th':   trial.suggest_float("tm_vol_th",   *narrow_bounds['tm_vol_th']),
#         'rsi_sell_th': trial.suggest_float("rsi_sell_th", *narrow_bounds['rsi_sell_th']),
#     })

# cma_study = optuna.create_study(direction="maximize",
#                                 sampler=optuna.samplers.CmaEsSampler())
# cma_study.optimize(objective_cma, n_trials=100)

# best_cma = cma_study.best_params
# print("▶ [CMA-ES] Best params:", best_cma, "ROI:", cma_study.best_value)

# # ────────────────────────────────────────────────────────────────
# # 7) 연구결과 합치기
# # ────────────────────────────────────────────────────────────────
# df_tpe = tpe_study.trials_dataframe().assign(stage='TPE')
# df_cma = cma_study.trials_dataframe().assign(stage='CMA-ES')
# all_trials = pd.concat([df_tpe, df_cma], ignore_index=True)
# all_trials.to_excel(os.path.join(results_folder,"two_stage_optimization.xlsx"),
#                     index=False)

# # ────────────────────────────────────────────────────────────────
# # 8) 누적 Best ROI 수렴 비교
# # ────────────────────────────────────────────────────────────────
# plt.figure(figsize=(8,6))
# for study,label in [(tpe_study,'TPE'), (cma_study,'CMA-ES')]:
#     vals = study.trials_dataframe()['value']
#     best_so_far = vals.cummax()
#     plt.plot(best_so_far.values, label=label)
# plt.xlabel('Trial Index')
# plt.ylabel('Best-so-far ROI (%)')
# plt.title('Two-Stage Optimization: Best-so-far ROI')
# plt.legend()
# plt.grid(True)
# plt.savefig(os.path.join(results_folder,"two_stage_convergence.png"),
#             dpi=300, bbox_inches='tight')
# plt.close()

# # ────────────────────────────────────────────────────────────────
# # 8) Trial 순서대로의 ROI 플롯 (Raw values)
# # ────────────────────────────────────────────────────────────────
# plt.figure(figsize=(8,6))
# for study, label in [(tpe_study, 'TPE'), (cma_study, 'CMA-ES')]:
#     # value 컬럼이 바로 각 트라이얼의 ROI
#     vals = study.trials_dataframe()['value']
#     plt.plot(
#         np.arange(len(vals)),
#         vals.values,
#         marker='o' if label=='TPE' else 'x',
#         linestyle='-' if label=='TPE' else '--',
#         alpha=0.8,
#         label=label
#     )

# plt.xlabel('Trial Index')
# plt.ylabel('ROI (%)')
# plt.title('Two-Stage Optimization: ROI per Trial')
# plt.legend()
# plt.grid(True)
# per_trial_plot = os.path.join(results_folder, "two_stage_roi_per_trial.png")
# plt.savefig(per_trial_plot, dpi=300, bbox_inches='tight')
# plt.close()

# print("✅ Trial 순서대로의 ROI 플롯 저장:", per_trial_plot)


# # ────────────────────────────────────────────────────────────────
# # 9) TPE 전용 파라미터 수렴 오버레이 플롯
# # ────────────────────────────────────────────────────────────────
# param_keys = list(global_bounds.keys())
# plt.figure(figsize=(12, 8))
# for p in param_keys:
#     tpe_vals = df_tpe[f'params_{p}'].values
#     plt.plot(
#         np.arange(len(tpe_vals)),
#         tpe_vals,
#         marker='o',
#         linestyle='-',
#         alpha=0.8,
#         label=p
#     )
# plt.xlabel('Trial Index')
# plt.ylabel('Parameter Value')
# plt.title('TPE Sampler: Parameter Convergence')
# plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
# plt.grid(True)
# plt.tight_layout()
# tpe_overlay = os.path.join(results_folder, "tpe_params_overlay.png")
# plt.savefig(tpe_overlay, dpi=300, bbox_inches='tight')
# plt.close()
# print("✅ TPE 파라미터 수렴 플롯 저장:", tpe_overlay)


# # ────────────────────────────────────────────────────────────────
# # 10) CMA-ES 전용 파라미터 수렴 오버레이 플롯
# # ────────────────────────────────────────────────────────────────
# plt.figure(figsize=(12, 8))
# for p in param_keys:
#     cma_vals = df_cma[f'params_{p}'].values
#     plt.plot(
#         np.arange(len(cma_vals)),
#         cma_vals,
#         marker='x',
#         linestyle='--',
#         alpha=0.8,
#         label=p
#     )
# plt.xlabel('Trial Index')
# plt.ylabel('Parameter Value')
# plt.title('CMA-ES Sampler: Parameter Convergence')
# plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
# plt.grid(True)
# plt.tight_layout()
# cma_overlay = os.path.join(results_folder, "cma_params_overlay.png")
# plt.savefig(cma_overlay, dpi=300, bbox_inches='tight')
# plt.close()
# print("✅ CMA-ES 파라미터 수렴 플롯 저장:", cma_overlay)

# # ────────────────────────────────────────────────────────────────
# # 11) Parameter Importances 계산 & 시각화
# # ────────────────────────────────────────────────────────────────
# from optuna.importance import get_param_importances

# # (1) 중요도 계산
# imp_tpe = get_param_importances(tpe_study)
# imp_cma = get_param_importances(cma_study)

# # (2) 콘솔 출력
# print("▶ [TPE] Parameter Importances:")
# for k, v in sorted(imp_tpe.items(), key=lambda x: -x[1]):
#     print(f"   {k}: {v:.3f}")

# print("\n▶ [CMA-ES] Parameter Importances:")
# for k, v in sorted(imp_cma.items(), key=lambda x: -x[1]):
#     print(f"   {k}: {v:.3f}")

# # (3) 바 차트 그리기 함수
# def plot_importance(imp_dict, title, filepath):
#     plt.figure(figsize=(6,4))
#     sns.barplot(x=list(imp_dict.values()), y=list(imp_dict.keys()), palette='viridis')
#     plt.title(title)
#     plt.xlabel('Importance')
#     plt.tight_layout()
#     plt.savefig(filepath, dpi=300)
#     plt.close()

# # (4) 차트 저장
# tpe_imp_png = os.path.join(results_folder, "tpe_param_importance.png")
# cma_imp_png = os.path.join(results_folder, "cma_param_importance.png")

# plot_importance(imp_tpe, "TPE Sampler Parameter Importances", tpe_imp_png)
# plot_importance(imp_cma, "CMA-ES Sampler Parameter Importances", cma_imp_png)

# print("✅ 중요도 플롯 저장 완료:")
# print("   •", tpe_imp_png)
# print("   •", cma_imp_png)


In [ ]:
#일반화

In [ ]:
# import os
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# import optuna
# from optuna.importance import get_param_importances
# from sklearn.preprocessing import MinMaxScaler

# # ────────────────────────────────────────────────────────────────
# # 1) 경로 설정
# # ────────────────────────────────────────────────────────────────
# input_csv      = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# macro_folder   = r"C:\Users\seung\OneDrive\주식\Macro Data"
# results_folder = r"C:\Users\seung\OneDrive\주식\Results"
# os.makedirs(results_folder, exist_ok=True)

# # ────────────────────────────────────────────────────────────────
# # 2) 데이터 로딩 및 전처리
# # ────────────────────────────────────────────────────────────────
# df = pd.read_csv(input_csv, encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2019-01-02') & (df['날짜'] < '2025-06-24')].reset_index(drop=True)

# vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"), parse_dates=[0], encoding='utf-8-sig')
# vix_df.columns = ['날짜','VIX']
# hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
#                      parse_dates=[0], encoding='utf-8-sig')
# hys_df.columns = ['날짜','HYS']

# df = df.merge(vix_df, on='날짜', how='left').merge(hys_df, on='날짜', how='left')
# df['거래량_log'] = np.log1p(df['거래량'])

# # ────────────────────────────────────────────────────────────────
# # 3) 전략 조건 클래스
# # ────────────────────────────────────────────────────────────────
# class StrategyCondition:
#     def check(self, row): raise NotImplementedError()

# class RSIBelow(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] < self.th

# class BollingerNearLower(StrategyCondition):
#     def __init__(self, buf): self.buf = buf
#     def check(self, row):
#         return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)

# class MACDPositive(StrategyCondition):
#     def check(self, row): return row['MACD'] > row['MACD 시그널']

# class MA5AboveMA10(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] > row['SMA 10일']

# class MA5BelowMA60(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] < row['SMA 60일']

# class TwoWeekPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (2주)'] < self.pct

# class ThreeMonthPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (3개월)'] < self.pct

# class TwoWeekVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (2주)'] < self.pct

# class ThreeMonthVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct

# class RSISell(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] > self.th

# # ────────────────────────────────────────────────────────────────
# # 4) 백테스트 함수 (ROI만 반환)
# # ────────────────────────────────────────────────────────────────
# def backtest_roi(params):
#     cash, shares = 10_000.0, 0.0
#     buy_conds = [
#         RSIBelow(params['rsi_buy_th']),
#         BollingerNearLower(params['boll_buffer']),
#         MACDPositive(),
#         MA5AboveMA10(),
#         MA5BelowMA60(),
#         TwoWeekPriceLow(params['tw_price_th']),
#         ThreeMonthPriceLow(params['tm_price_th']),
#         TwoWeekVolumeLow(params['tw_vol_th']),
#         ThreeMonthVolumeLow(params['tm_vol_th']),
#     ]
#     sell_conds = [RSISell(params['rsi_sell_th'])]

#     for _, row in df.iterrows():
#         price, vix = row['종가'], row['VIX']
#         # 매크로 강제 매수
#         if shares == 0 and not pd.isna(vix) and vix >= 60:
#             shares, cash = cash/price, 0.0
#             continue
#         # 룰 기반 매수
#         if shares == 0 and all(cond.check(row) for cond in buy_conds):
#             shares, cash = cash/price, 0.0
#         # 룰 기반 매도
#         elif shares > 0 and any(cond.check(row) for cond in sell_conds):
#             cash, shares = shares*price, 0.0

#     final_val = cash + shares * df.iloc[-1]['종가']
#     return (final_val - 10_000.0) / 10_000.0 * 100

# # ────────────────────────────────────────────────────────────────
# # 5) 1단계: TPE로 빠른 탐색
# # ────────────────────────────────────────────────────────────────
# def objective_tpe(trial):
#     return backtest_roi({
#         'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
#         'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
#         'tw_price_th': trial.suggest_float("tw_price_th", 0, 20),
#         'tm_price_th': trial.suggest_float("tm_price_th", 0, 50),
#         'tw_vol_th':   trial.suggest_float("tw_vol_th",   0, 100),
#         'tm_vol_th':   trial.suggest_float("tm_vol_th",   0, 300),
#         'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
#     })

# tpe_study = optuna.create_study(direction="maximize",
#                                 sampler=optuna.samplers.TPESampler())
# tpe_study.optimize(objective_tpe, n_trials=500)
# best_tpe = tpe_study.best_params
# print("▶ [TPE] Best params:", best_tpe, "ROI:", tpe_study.best_value)

# # ────────────────────────────────────────────────────────────────
# # 6) 중요도 계산 & 자동 파라미터 필터링
# # ────────────────────────────────────────────────────────────────
# imp_tpe = get_param_importances(tpe_study)
# threshold = 0.05
# important_params = [k for k, v in imp_tpe.items() if v >= threshold]
# print(f"▶ Parameter importances (TPE): {imp_tpe}")
# print(f"▶ Importance ≥ {threshold}: {important_params}")

# # ────────────────────────────────────────────────────────────────
# # 7) 2단계: CMA-ES로 정밀 탐색 (중요한 파라미터만)
# # ────────────────────────────────────────────────────────────────
# global_bounds = {
#     'rsi_buy_th':   (0,100),
#     'boll_buffer':  (0,0.1),
#     'tw_price_th':  (0,20),
#     'tm_price_th':  (0,50),
#     'tw_vol_th':    (0,100),
#     'tm_vol_th':    (0,300),
#     'rsi_sell_th':  (0,100),
# }

# narrow_bounds = {}
# for k in important_params:
#     lo, hi = global_bounds[k]
#     bp = best_tpe[k]
#     delta = 0.2 * (hi - lo)
#     narrow_bounds[k] = (max(lo, bp - delta), min(hi, bp + delta))

# def objective_cma(trial):
#     p = {}
#     for k in global_bounds:
#         if k in important_params:
#             lo, hi = narrow_bounds[k]
#             p[k] = trial.suggest_float(k, lo, hi)
#         else:
#             p[k] = best_tpe[k]
#     return backtest_roi(p)

# cma_study = optuna.create_study(direction="maximize",
#                                 sampler=optuna.samplers.CmaEsSampler())
# cma_study.optimize(objective_cma, n_trials=500)
# best_cma = cma_study.best_params
# print("▶ [CMA-ES] Best params:", best_cma, "ROI:", cma_study.best_value)

# # ────────────────────────────────────────────────────────────────
# # 8) 결과 저장 및 시각화
# # ────────────────────────────────────────────────────────────────
# df_tpe = tpe_study.trials_dataframe().assign(stage='TPE')
# df_cma = cma_study.trials_dataframe().assign(stage='CMA-ES')
# pd.concat([df_tpe, df_cma], ignore_index=True) \
#   .to_excel(os.path.join(results_folder, "two_stage_auto_filter.xlsx"), index=False)

# # ROI per trial
# plt.figure(figsize=(8,6))
# for study,label,marker in [(tpe_study,'TPE','o'), (cma_study,'CMA-ES','x')]:
#     vals = study.trials_dataframe()['value']
#     plt.plot(range(len(vals)), vals.values, marker=marker, label=label)
# plt.xlabel('Trial'); plt.ylabel('ROI (%)')
# plt.title('ROI per Trial'); plt.legend(); plt.grid(True)
# plt.savefig(os.path.join(results_folder,"two_stage_roi_per_trial.png"), dpi=300, bbox_inches='tight')
# plt.close()

# # TPE 파라미터 수렴 (오버레이)
# plt.figure(figsize=(12,8))
# for k in global_bounds.keys():
#     plt.plot(df_tpe[f'params_{k}'], label=k)
# plt.title('TPE Parameter Convergence')
# plt.xlabel('Trial'); plt.ylabel('Value')
# plt.legend(bbox_to_anchor=(1.01,1), loc='upper left'); plt.grid(True)
# plt.tight_layout()
# plt.savefig(os.path.join(results_folder,"tpe_params_overlay.png"), dpi=300, bbox_inches='tight')
# plt.close()

# # CMA-ES 파라미터 수렴 (중요 파라미터만)
# plt.figure(figsize=(12,8))
# for k in important_params:
#     plt.plot(df_cma[f'params_{k}'], label=k)
# plt.title('CMA-ES Parameter Convergence')
# plt.xlabel('Trial'); plt.ylabel('Value')
# plt.legend(bbox_to_anchor=(1.01,1), loc='upper left'); plt.grid(True)
# plt.tight_layout()
# plt.savefig(os.path.join(results_folder,"cma_params_overlay.png"), dpi=300, bbox_inches='tight')
# plt.close()

# print("✅ Two-stage optimization with auto-filter complete.")
# print("  • Results:", os.path.join(results_folder, "two_stage_auto_filter.xlsx"))
# print("  • ROI per trial plot saved.")
# print("  • Parameter convergence plots saved.")


In [ ]:
# VIX 하한 상항 강제 매수 매도 boundary도 parameter로 포함시킨 optimziation

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.importance import get_param_importances

# ───────────────────────────────────────────────────────────────
# 1) 경로 설정
# ───────────────────────────────────────────────────────────────
input_csv      = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\seung\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\seung\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# ───────────────────────────────────────────────────────────────
# 2) 데이터 로딩 및 전처리
# ───────────────────────────────────────────────────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2019-01-02') & (df['날짜'] < '2025-06-24')].reset_index(drop=True)

vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"), parse_dates=[0], encoding='utf-8-sig')
vix_df.columns = ['날짜','VIX']
hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
                     parse_dates=[0], encoding='utf-8-sig')
hys_df.columns = ['날짜','HYS']

df = df.merge(vix_df, on='날짜', how='left').merge(hys_df, on='날짜', how='left')
df['거래량_log'] = np.log1p(df['거래량'])

# ───────────────────────────────────────────────────────────────
# 3) 전략 조건 클래스
# ───────────────────────────────────────────────────────────────
class StrategyCondition:
    def check(self, row): raise NotImplementedError()

class RSIBelow(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] < self.th

class BollingerNearLower(StrategyCondition):
    def __init__(self, buf): self.buf = buf
    def check(self, row):
        return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)

class MACDPositive(StrategyCondition):
    def check(self, row): return row['MACD'] > row['MACD 시그널']

class MA5AboveMA10(StrategyCondition):
    def check(self, row): return row['SMA 5일'] > row['SMA 10일']

class MA5BelowMA60(StrategyCondition):
    def check(self, row): return row['SMA 5일'] < row['SMA 60일']

class TwoWeekPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (2주)'] < self.pct

class ThreeMonthPriceLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['가격 상승률 (3개월)'] < self.pct

class TwoWeekVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (2주)'] < self.pct

class ThreeMonthVolumeLow(StrategyCondition):
    def __init__(self, pct): self.pct = pct
    def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct

class RSISell(StrategyCondition):
    def __init__(self, th): self.th = th
    def check(self, row): return row['RSI (14일)'] > self.th

# ───────────────────────────────────────────────────────────────
# 4) 백테스트 함수 (매크로: VIX 상/하한, 기술적 룰)
# ───────────────────────────────────────────────────────────────
def backtest_roi(params):
    cash, shares = 10_000.0, 0.0

    buy_conds = [
        RSIBelow(params['rsi_buy_th']),
        BollingerNearLower(params['boll_buffer']),
        MACDPositive(),
        MA5AboveMA10(),
        MA5BelowMA60(),
        TwoWeekPriceLow(params['tw_price_th']),
        ThreeMonthPriceLow(params['tm_price_th']),
        TwoWeekVolumeLow(params['tw_vol_th']),
        ThreeMonthVolumeLow(params['tm_vol_th']),
    ]
    sell_conds = [RSISell(params['rsi_sell_th'])]

    for _, row in df.iterrows():
        price, vix = row['종가'], row['VIX']

        # 1) VIX 상한 넘으면 강제 매수
        if shares == 0 and not pd.isna(vix) and vix >= params['vix_buy_th']:
            shares, cash = cash/price, 0.0
            continue

        # 2) VIX 하한 밑이면 강제 매도
        if shares > 0 and not pd.isna(vix) and vix <= params['vix_sell_th']:
            cash, shares = shares*price, 0.0
            continue

        # 3) 기술적 룰 매수
        if shares == 0 and all(cond.check(row) for cond in buy_conds):
            shares, cash = cash/price, 0.0

        # 4) 기술적 룰 매도
        elif shares > 0 and any(cond.check(row) for cond in sell_conds):
            cash, shares = shares*price, 0.0

    final_val = cash + shares * df.iloc[-1]['종가']
    return (final_val - 10_000.0) / 10_000.0 * 100

# ───────────────────────────────────────────────────────────────
# 5) 1단계: TPE로 빠른 탐색
# ───────────────────────────────────────────────────────────────
def objective_tpe(trial):
    vb = trial.suggest_float("vix_buy_th",  0.0, 100.0)
    vs = trial.suggest_float("vix_sell_th", 0.0, vb)
    return backtest_roi({
        'vix_buy_th':  vb,
        'vix_sell_th': vs,
        'rsi_buy_th':  trial.suggest_float("rsi_buy_th",  0, 100),
        'boll_buffer': trial.suggest_float("boll_buffer", 0, 0.1),
        'tw_price_th': trial.suggest_float("tw_price_th", 0, 20),
        'tm_price_th': trial.suggest_float("tm_price_th", 0, 50),
        'tw_vol_th':   trial.suggest_float("tw_vol_th",   0, 100),
        'tm_vol_th':   trial.suggest_float("tm_vol_th",   0, 300),
        'rsi_sell_th': trial.suggest_float("rsi_sell_th", 0, 100),
    })

tpe_study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler())
tpe_study.optimize(objective_tpe, n_trials=1000)

best_tpe = tpe_study.best_params
print("▶ [TPE] Best ROI:", round(tpe_study.best_value,2))
print("▶ [TPE] Best Parameters:")
for k, v in best_tpe.items():
    print(f"    • {k}: {v}")

# ───────────────────────────────────────────────────────────────
# 6) 중요도 계산 & 자동 파라미터 필터링
# ───────────────────────────────────────────────────────────────
imp = get_param_importances(tpe_study)
thr = 0.05
important = [k for k,v in imp.items() if v>=thr]
print("\n▶ [TPE] Parameter Importances:")
for k,v in imp.items():
    print(f"    • {k}: {v:.3f}")
print(f"▶ Keep ≥ {thr}: {important}")

# ───────────────────────────────────────────────────────────────
# 7) 2단계: CMA-ES로 정밀 탐색 (중요 파라미터만, 나머 fixed)
# ───────────────────────────────────────────────────────────────
bounds = {
  "vix_buy_th":  (0.0,100.0),
  "vix_sell_th": (0.0,100.0),
  "rsi_buy_th":  (0.0,100.0),
  "boll_buffer": (0.0,0.1),
  "tw_price_th": (0.0,20.0),
  "tm_price_th": (0.0,50.0),
  "tw_vol_th":   (0.0,100.0),
  "tm_vol_th":   (0.0,300.0),
  "rsi_sell_th": (0.0,100.0),
}
narrow = {}
for k in important:
    lo,hi = bounds[k]
    bp = best_tpe[k]
    d = 0.2*(hi-lo)
    narrow[k] = (max(lo,bp-d), min(hi,bp+d))

def objective_cma(trial):
    p = {}
    for k,(lo,hi) in bounds.items():
        if k in important:
            lo2,hi2 = narrow[k]
            if k=="vix_buy_th":
                val = trial.suggest_float(k, lo2, hi2)
            elif k=="vix_sell_th":
                val = trial.suggest_float(k, lo2, min(hi2, best_tpe["vix_buy_th"]))
            else:
                val = trial.suggest_float(k, lo2, hi2)
            p[k] = val
        else:
            p[k] = best_tpe[k]
    return backtest_roi(p)

cma_study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.CmaEsSampler())
cma_study.optimize(objective_cma, n_trials=1000)

best_cma = cma_study.best_params
print("\n▶ [CMA-ES] Best ROI:", round(cma_study.best_value,2))
print("▶ [CMA-ES] Best Parameters:")
for k, v in best_cma.items():
    print(f"    • {k}: {v}")

# ───────────────────────────────────────────────────────────────

# ───────────────────────────────────────────────────────────────

# 8) 결과 합치기 & 저장
df_tpe = tpe_study.trials_dataframe().assign(stage="TPE")
df_cma = cma_study.trials_dataframe().assign(stage="CMA-ES")
out_path = os.path.join(results_folder,"two_stage_with_vix.xlsx")
pd.concat([df_tpe,df_cma],ignore_index=True)\
  .to_excel(out_path, index=False)
print("\n✅ 저장 완료 →", out_path)

# ───────────────────────────────────────────────────────────────
# 9) 최종(best ROI) 조합 출력
#    TPE 에서 고정된 파라미터(best_tpe) + CMA-ES 로 튜닝된 파라미터(best_cma)
# ───────────────────────────────────────────────────────────────
# TPE, CMA-ES 각각의 best params
print("\n▶ [TPE] Best ROI:", round(tpe_study.best_value,2))
print("▶ [TPE] Best Parameters:")
for k, v in best_tpe.items():
    print(f"   • {k}: {v}")

print("\n▶ [CMA-ES] Best ROI:", round(cma_study.best_value,2))
print("▶ [CMA-ES] Best Parameters:")
for k, v in best_cma.items():
    print(f"   • {k}: {v}")

# TPE 파라미터에 CMA-ES 결과 덮어씌워서 “최종 조합” 생성
full_best = best_tpe.copy()
full_best.update(best_cma)

print("\n▶ [Final Combined] Best ROI:", round(cma_study.best_value,2))
print("▶ [Final Combined] All Parameters:")
for k, v in full_best.items():
    print(f"   • {k}: {v}")

In [ ]:
▶ [CMA-ES] Best params: {'vix_buy_th': 35.47537612506862, 'rsi_sell_th': 89.45936968684146} ROI: 5709.160379021657
✅ Done. 결과: two_stage_with_vix.xlsx, 플롯들 → Results 폴더
    
    ▶ [TPE] Best ROI: 5588.42
▶ [TPE] Best Parameters:
    • vix_buy_th: 34.78529194184235
    • vix_sell_th: 5.799125806917302
    • rsi_buy_th: 78.49990960271758
    • boll_buffer: 0.09570817361851704
    • tw_price_th: 17.28209132296718
    • tm_price_th: 11.203875933889309
    • tw_vol_th: 14.576505184329823
    • tm_vol_th: 8.742939261779691
    • rsi_sell_th: 89.82169909044313

In [ ]:
2020년부터 2024년 12월까지

⭐️ Best ROI (%): 357878.73
⭐️ Best Params:
   rsi_buy_th: 86.4067
   boll_buffer: 0.0415
   tw_price_th: 3.3351
   tm_price_th: 15.1196
   tw_vol_th: 43.4293
   tm_vol_th: 217.0325
   rsi_sell_th: 96.3093
▶ Validation ROI with best params: 357878.73%
    

⭐️ Best ROI (%): 269501.09
⭐️ Best Params:
   rsi_buy_th: 47.8733
   boll_buffer: 0.0150
   tw_price_th: 10.2913
   tm_price_th: 23.1772
   tw_vol_th: 70.1934
   tm_vol_th: 225.2615
   rsi_sell_th: 96.8699
▶ Validation ROI with best params: 269501.09%


VIX < 60 with 100

⭐️ Best ROI (%): 221461.74
⭐️ Best Params:
   rsi_buy_th: 51.1987
   boll_buffer: 0.0611
   tw_price_th: 0.1096
   tm_price_th: 33.3657
   tw_vol_th: 83.8030
   tm_vol_th: 48.7716
   rsi_sell_th: 96.9792
▶ Validation ROI with best params: 221461.74%



VIX 30 < <60 with 1000

⭐️ Best ROI (%): 15123.81
⭐️ Best Params:
   rsi_buy_th: 86.3576
   boll_buffer: 0.0938
   tw_price_th: 4.8323
   tm_price_th: 15.0876
   tw_vol_th: 16.6268
   tm_vol_th: 153.2832
   rsi_sell_th: 39.6654
▶ Validation ROI with best params: 15123.81%


VIX 30 < <60 with 100
⭐️ Best ROI (%): 14669.3
⭐️ Best Params:
   rsi_buy_th: 81.4464
   boll_buffer: 0.0898
   tw_price_th: 4.6479
   tm_price_th: 29.1587
   tw_vol_th: 16.4856
   tm_vol_th: 8.1081
   rsi_sell_th: 23.2581
▶ Validation ROI with best params: 14669.30%
    
VIX 30< < 50
# # How much portion to sell and buy optimization
⭐️ Best ROI (%): 131961.59
⭐️ Best Params:
   rsi_buy_th: 32.9508
   boll_buffer: 0.0844
   tw_price_th: 4.0799
   tm_price_th: 27.3716
   tw_vol_th: 43.3509
   tm_vol_th: 133.7565
   rsi_sell_th: 93.5607
▶ Validation ROI with best params: 131961.59%

⭐️ Best ROI (%): 8189.49
⭐️ Best Params:
   rsi_buy_th: 93.3247
   boll_buffer: 0.0888
   tw_price_th: 4.9123
   tm_price_th: 31.7569
   tw_vol_th: 17.0601
   tm_vol_th: 9.6545
   rsi_sell_th: 47.2580
▶ Validation ROI with best params: 8189.49%

# 매번 buy signal 뜰때마다 10,000달러 더 한다. + sell portion of current position

In [ ]:
#이 코드 써

In [ ]:
# import os
# import pandas as pd
# import numpy as np

# # 1) 최적 파라미터 (Optuna 결과)
# params = {
#     'rsi_buy_th': 86.4067,
#     'boll_buffer': 0.0415,
#     'tw_price_th': 3.3351,
#     'tm_price_th': 15.1196,
#     'tw_vol_th': 43.4293,
#     'tm_vol_th': 217.0325,
#     'rsi_sell_th': 96.3093
# }

# # 2) 경로 설정 (환경에 맞게 수정)
# input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# macro_vix      = r"C:\Users\LabPC\OneDrive\주식\Macro Data\VIX.csv"
# macro_hys      = r"C:\Users\LabPC\OneDrive\주식\Macro Data\High Yied Spread adjusted.csv"
# results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
# os.makedirs(results_folder, exist_ok=True)

# # 3) 데이터 로드 & 병합
# df = pd.read_csv(input_csv, encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# vix = pd.read_csv(macro_vix, parse_dates=[0], encoding='utf-8-sig')
# vix.columns = ['날짜','VIX']
# hys = pd.read_csv(macro_hys, parse_dates=[0], encoding='utf-8-sig')
# hys.columns = ['날짜','HYS']

# df = df.merge(vix, on='날짜', how='left').merge(hys, on='날짜', how='left')

# # 4) 전략 조건 클래스
# class StrategyCondition:
#     def check(self, row): raise NotImplementedError()
# class RSIBelow(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] < self.th
# class BollingerNearLower(StrategyCondition):
#     def __init__(self, buf): self.buf = buf
#     def check(self, row): return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)
# class MACDPositive(StrategyCondition):
#     def check(self, row): return row['MACD'] > row['MACD 시그널']
# class MA5AboveMA10(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] > row['SMA 10일']
# class MA5BelowMA60(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] < row['SMA 60일']
# class TwoWeekPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (2주)'] < self.pct
# class ThreeMonthPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (3개월)'] < self.pct
# class TwoWeekVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (2주)'] < self.pct
# class ThreeMonthVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct
# class RSISell(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] > self.th

# # 5) 백테스트 + 로그 기록
# def backtest_with_log(params, extra_on_buy=False):
#     initial = 10000.0
#     cash, shares = initial, 0.0
#     total_invested = initial
#     logs = []

#     buy_conds = [
#         RSIBelow(params['rsi_buy_th']),
#         BollingerNearLower(params['boll_buffer']),
#         MACDPositive(),
#         MA5AboveMA10(),
#         MA5BelowMA60(),
#         TwoWeekPriceLow(params['tw_price_th']),
#         ThreeMonthPriceLow(params['tm_price_th']),
#         TwoWeekVolumeLow(params['tw_vol_th']),
#         ThreeMonthVolumeLow(params['tm_vol_th']),
#     ]
#     sell_conds = [RSISell(params['rsi_sell_th'])]

#     for _, row in df.iterrows():
#         date, price, vix = row['날짜'], row['종가'], row['VIX']

#         # 1) 매크로 강제 매수 신호
#         if shares == 0 and not pd.isna(vix) and vix >= 60:
#             invest = 10000.0 if extra_on_buy else cash
#             if invest <= 0: continue
#             cash += invest if extra_on_buy else 0.0
#             total_invested += invest if extra_on_buy else 0.0
#             bought = invest / price
#             shares += bought
#             cash -= invest
#             asset = cash + shares*price
#             roi   = (asset - initial) / initial * 100
#             logs.append([date, "BUY_MACRO", price, bought, cash, asset, roi])
#             continue

#         # 2) 룰베이스 매수
#         if shares == 0 and all(cond.check(row) for cond in buy_conds):
#             invest = 10000.0 if extra_on_buy else cash
#             if invest <= 0: continue
#             cash += invest if extra_on_buy else 0.0
#             total_invested += invest if extra_on_buy else 0.0
#             bought = invest / price
#             shares += bought
#             cash -= invest
#             asset = cash + shares*price
#             roi   = (asset - initial) / initial * 100
#             logs.append([date, "BUY", price, bought, cash, asset, roi])

#         # 3) 룰베이스 매도
#         elif shares > 0 and any(cond.check(row) for cond in sell_conds):
#             sold = shares
#             cash += sold * price
#             shares = 0.0
#             asset = cash
#             roi   = (asset - initial) / initial * 100
#             logs.append([date, "SELL", price, sold, cash, asset, roi])

#     # 4) 최종 청산 기록 (FINAL)
#     final_price = df.iloc[-1]['종가']
#     date = df.iloc[-1]['날짜']
#     if shares > 0:
#         cash += shares * final_price
#         shares = 0.0
#     asset = cash
#     roi   = (asset - initial) / initial * 100
#     logs.append([date, "FINAL", final_price, 0.0, cash, asset, roi])

#     return pd.DataFrame(logs, columns=["날짜","액션","가격","수량","현금","총자산","ROI(%)"])

# # 6) 실행 후 CSV 저장
# log_one   = backtest_with_log(params, extra_on_buy=False)
# log_extra = backtest_with_log(params, extra_on_buy=True)

# log_one  .to_csv(os.path.join(results_folder, "log_one_time_10000.csv"),     index=False, encoding='utf-8-sig')
# log_extra.to_csv(os.path.join(results_folder, "log_extra_10000_each_buy.csv"),index=False, encoding='utf-8-sig')

# print("✅ 파일 저장 완료:")
# print("   •", os.path.join(results_folder, "log_one_time_10000.csv"))
# print("   •", os.path.join(results_folder, "log_extra_10000_each_buy.csv"))


In [ ]:
# import os
# import pandas as pd
# import numpy as np

# # ─── 1) 최적 파라미터 ─────────────────────────────────────────
# params = {
#     'rsi_buy_th': 86.4067,
#     'boll_buffer': 0.0415,
#     'tw_price_th': 3.3351,
#     'tm_price_th': 15.1196,
#     'tw_vol_th': 43.4293,
#     'tm_vol_th': 217.0325,
#     'rsi_sell_th': 96.3093
# }

# # ─── 2) 경로 설정 (환경에 맞게 수정) ───────────────────────────
# # input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# # macro_vix      = r"C:\Users\LabPC\OneDrive\주식\Macro Data\VIX.csv"
# # macro_hys      = r"C:\Users\LabPC\OneDrive\주식\Macro Data\High Yied Spread adjusted.csv"
# # results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"


# input_csv      = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# macro_vix      = r"C:\Users\seung\OneDrive\주식\Macro Data\VIX.csv"
# macro_hys      = r"C:\Users\seung\OneDrive\주식\Macro Data\High Yied Spread adjusted.csv"
# results_folder = r"C:\Users\seung\OneDrive\주식\Results"

# os.makedirs(results_folder, exist_ok=True)

# # ─── 3) 데이터 로드 & 병합 ─────────────────────────────────────
# df = pd.read_csv(input_csv, encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-06-05')].reset_index(drop=True)
# vix = pd.read_csv(macro_vix, parse_dates=[0], encoding='utf-8-sig'); vix.columns = ['날짜','VIX']
# hys = pd.read_csv(macro_hys, parse_dates=[0], encoding='utf-8-sig'); hys.columns = ['날짜','HYS']
# df = df.merge(vix, on='날짜', how='left').merge(hys, on='날짜', how='left')

# # ─── 4) 전략 조건 클래스 ────────────────────────────────────
# class StrategyCondition:
#     def check(self, row): raise NotImplementedError()
# class RSIBelow(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] < self.th
# class BollingerNearLower(StrategyCondition):
#     def __init__(self, buf): self.buf = buf
#     def check(self, row): return row['종가'] < row['볼린저밴드 하단'] * (1 + self.buf)
# class MACDPositive(StrategyCondition):
#     def check(self, row): return row['MACD'] > row['MACD 시그널']
# class MA5AboveMA10(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] > row['SMA 10일']
# class MA5BelowMA60(StrategyCondition):
#     def check(self, row): return row['SMA 5일'] < row['SMA 60일']
# class TwoWeekPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (2주)'] < self.pct
# class ThreeMonthPriceLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['가격 상승률 (3개월)'] < self.pct
# class TwoWeekVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (2주)'] < self.pct
# class ThreeMonthVolumeLow(StrategyCondition):
#     def __init__(self, pct): self.pct = pct
#     def check(self, row): return row['거래량 상승률 (3개월)'] < self.pct
# class RSISell(StrategyCondition):
#     def __init__(self, th): self.th = th
#     def check(self, row): return row['RSI (14일)'] > self.th

# # ─── 5) 백테스트 + 로그 기록 (첫 매수만 초기 자금 사용) ─────────────
# def backtest_with_log(params, extra_on_buy=False):
#     initial = 10000.0
#     cash, shares = initial, 0.0
#     total_invested = initial
#     first_trade = True
#     logs = []

#     buy_conds  = [
#         RSIBelow(params['rsi_buy_th']),
#         BollingerNearLower(params['boll_buffer']),
#         MACDPositive(),
#         MA5AboveMA10(),
#         MA5BelowMA60(),
#         TwoWeekPriceLow(params['tw_price_th']),
#         ThreeMonthPriceLow(params['tm_price_th']),
#         TwoWeekVolumeLow(params['tw_vol_th']),
#         ThreeMonthVolumeLow(params['tm_vol_th']),
#     ]
#     sell_conds = [RSISell(params['rsi_sell_th'])]

#     for _, row in df.iterrows():
#         date, price, vix = row['날짜'], row['종가'], row['VIX']

#         # ── A) 매크로 강제 매수 ──
#         if shares == 0 and not pd.isna(vix) and vix >= 60:
#             # 첫 매수인 경우, 초기 금액만 사용
#             if first_trade:
#                 invest = cash
#             else:
#                 invest = 10000.0 if extra_on_buy else cash
#             # 외부 자금 투입
#             if extra_on_buy and not first_trade:
#                 cash += 10000.0
#                 total_invested += 10000.0
#             shares += invest / price
#             cash -= invest
#             asset = cash + shares * price
#             roi = (asset - initial) / initial * 100
#             logs.append([date, "BUY_MACRO", price, invest/price, cash, asset, roi])
#             first_trade = False
#             continue

#         # ── B) 룰 기반 매수 ──
#         if shares == 0 and all(cond.check(row) for cond in buy_conds):
#             if first_trade:
#                 invest = cash
#             else:
#                 invest = 10000.0 if extra_on_buy else cash
#             if extra_on_buy and not first_trade:
#                 cash += 10000.0
#                 total_invested += 10000.0
#             shares += invest / price
#             cash -= invest
#             asset = cash + shares * price
#             roi = (asset - initial) / initial * 100
#             logs.append([date, "BUY", price, invest/price, cash, asset, roi])
#             first_trade = False

#         # ── C) 룰 기반 매도 ──
#         elif shares > 0 and any(cond.check(row) for cond in sell_conds):
#             sold = shares
#             cash += sold * price
#             shares = 0.0
#             asset = cash
#             roi = (asset - initial) / initial * 100
#             logs.append([date, "SELL", price, sold, cash, asset, roi])

#     # ── D) FINAL 청산 로그 ──
#     final_price = df.iloc[-1]['종가']
#     final_date  = df.iloc[-1]['날짜']
#     if shares > 0:
#         cash += shares * final_price
#         shares = 0.0
#     asset = cash
#     roi   = (asset - initial) / initial * 100
#     logs.append([final_date, "FINAL", final_price, 0.0, cash, asset, roi])

#     return pd.DataFrame(
#         logs,
#         columns=["날짜","액션","가격","수량","현금","총자산","ROI(%)"]
#     )

# # ─── 6) 실행 & CSV 저장 ───────────────────────────────────────────
# log_one   = backtest_with_log(params, extra_on_buy=False)
# log_extra = backtest_with_log(params, extra_on_buy=True)

# log_one  .to_csv(os.path.join(results_folder, "log_one_time_10000.csv"),     index=False, encoding='utf-8-sig')
# log_extra.to_csv(os.path.join(results_folder, "log_extra_10000_each_buy.csv"),index=False, encoding='utf-8-sig')

# print("✅ 결과 저장:")
# print("  •", os.path.join(results_folder, "log_one_time_10000.csv"))
# print("  •", os.path.join(results_folder, "log_extra_10000_each_buy.csv"))


In [ ]:
# import os
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns

# # 0) — 여기에 최적화된 파라미터를 통째로 붙여넣으세요 —
# param_str = """
#    rsi_buy_th: 86.3576
#    boll_buffer: 0.0938
#    tw_price_th: 4.8323
#    tm_price_th: 15.0876
#    tw_vol_th: 16.6268
#    tm_vol_th: 153.2832
#    rsi_sell_th: 39.6654
# """

# # 1) 파라미터 파싱
# params = {}
# for line in param_str.strip().splitlines():
#     key, val = line.split(':')
#     params[key.strip()] = float(val.strip())

# rsi_buy_th  = params['rsi_buy_th']
# boll_buffer = params['boll_buffer']
# tw_price_th = params['tw_price_th']
# tm_price_th = params['tm_price_th']
# tw_vol_th   = params['tw_vol_th']
# tm_vol_th   = params['tm_vol_th']
# rsi_sell_th = params['rsi_sell_th']

# # 2) 경로 설정
# input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
# results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"
# os.makedirs(results_folder, exist_ok=True)

# # 3) 데이터 로딩 및 필터링
# df = pd.read_csv(input_csv, encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

# # 4) 매크로 데이터 불러와 병합
# vix_df = pd.read_csv(os.path.join(macro_folder, "VIX.csv"),
#                      parse_dates=[0], encoding='utf-8-sig')
# vix_df.columns = ['날짜','VIX']
# hys_df = pd.read_csv(os.path.join(macro_folder, "High Yied Spread adjusted.csv"),
#                      parse_dates=[0], encoding='utf-8-sig')
# hys_df.columns = ['날짜','HYS']
# df = df.merge(vix_df, on='날짜', how='left')\
#        .merge(hys_df, on='날짜', how='left')

# # 5) 거래량 로그 변환 (기술적 룰에 사용)
# df['거래량_log'] = np.log1p(df['거래량'])

# # 6) 백테스트 함수 (Fraction 최적화용, 매수 시마다 10k + sell_portion 투입)
# def backtest(buy_frac, sell_frac):
#     cash = 0.0
#     shares = 0.0
#     total_injected = 0.0

#     for _, row in df.iterrows():
#         price = row['종가']

#         # ── 매크로 “강제 매도”
#         if shares > 0 and (
#             (not pd.isna(row['VIX']) and row['VIX'] >= 50) or
#             (not pd.isna(row['HYS']) and row['HYS'] >= 7)
#         ):
#             to_sell = shares * sell_frac
#             cash   += to_sell * price
#             shares -= to_sell
#             continue

#         # ── 매크로 “강제 매수”
#         if shares == 0 and (not pd.isna(row['VIX']) and row['VIX'] <= 30):
#             # inject fresh 10k
#             cash += 10_000.0
#             total_injected += 10_000.0
#             # invest fraction
#             invest = cash * buy_frac
#             shares += invest / price
#             cash -= invest
#             continue

#         # ── 기술적 룰 매수
#         if all([
#             shares >= 0,
#             row['RSI (14일)']           <  rsi_buy_th,
#             row['종가']                 <  row['볼린저밴드 하단'] * (1 + boll_buffer),
#             row['MACD']                 >  row['MACD 시그널'],
#             row['SMA 5일']              >  row['SMA 10일'],
#             row['SMA 5일']              <  row['SMA 60일'],
#             row['가격 상승률 (2주)']     <  tw_price_th,
#             row['가격 상승률 (3개월)']   <  tm_price_th,
#             row['거래량 상승률 (2주)']   <  tw_vol_th,
#             row['거래량 상승률 (3개월)'] <  tm_vol_th
#         ]):
#             # inject fresh 10k + sell portion of existing shares
#             cash += 10_000.0
#             total_injected += 10_000.0
#             # sell portion of current shares to raise extra cash
#             to_sell = shares * sell_frac
#             cash   += to_sell * price
#             shares -= to_sell
#             # invest fraction of total cash
#             invest = cash * buy_frac
#             shares += invest / price
#             cash -= invest

#         # ── 기술적 룰 매도
#         elif shares > 0 and row['RSI (14일)'] > rsi_sell_th:
#             to_sell = shares * sell_frac
#             cash   += to_sell * price
#             shares -= to_sell

#     # 최종 청산
#     final_value = cash + shares * df.iloc[-1]['종가']
#     # ROI 는 총 투입 금액 대비 수익률
#     roi = (final_value - total_injected) / total_injected * 100
#     return roi

# # 7) Fraction 그리드 탐색 & 결과 수집
# fracs   = [i/10 for i in range(1, 11)]
# records = []
# for buy in fracs:
#     for sell in fracs:
#         roi = backtest(buy, sell)
#         records.append({'buy_frac': buy, 'sell_frac': sell, 'ROI (%)': round(roi, 2)})

# # 8) DataFrame 변환 → CSV 저장
# out_df   = pd.DataFrame(records)
# csv_path = os.path.join(results_folder, "macro_rule_fraction_results.csv")
# out_df.to_csv(csv_path, index=False, encoding='utf-8-sig')
# print("✅ CSV 저장 완료:", csv_path)

# # 9) Heatmap 그리기 → PNG 저장
# heatmap_data = out_df.pivot(index='sell_frac', columns='buy_frac', values='ROI (%)')

# plt.figure(figsize=(8, 6))
# sns.heatmap(
#     heatmap_data,
#     annot=True, fmt=".2f", cmap='viridis',
#     cbar_kws={'label': 'ROI (%)'}
# )
# plt.title('Macro+Rule Strategy: Buy/Sell Fraction별 ROI Heatmap')
# plt.xlabel('Buy Fraction')
# plt.ylabel('Sell Fraction')

# png_path = os.path.join(results_folder, "macro_rule_fraction_heatmap.png")
# plt.savefig(png_path, dpi=300, bbox_inches='tight')
# plt.close()
# print("✅ Heatmap PNG 저장 완료:", png_path)

# # ──────────────────────────────────────────────────────────────
# # This is not financial advice, only data analysis.
# # Please consult a qualified financial professional for personalized guidance.


In [ ]:
# Given the optimized parameter and optimized sell & buy fraction. 
# give me the full simulation result

In [ ]:
# 

In [ ]:
### ojbect oreineted


In [ ]:
# 모든거 업데이트

In [ ]:
⭐️ Best ROI (%): 269501.09
⭐️ Best Params:
   rsi_buy_th: 47.8733
   boll_buffer: 0.0150
   tw_price_th: 10.2913
   tm_price_th: 23.1772
   tw_vol_th: 70.1934
   tm_vol_th: 225.2615
   rsi_sell_th: 96.8699
▶ Validation ROI with best params: 269501.09%

In [ ]:
# import pandas as pd
# import numpy as np
# import os

# # ── 1. 파라미터 입력 ──
# params = {
#     'rsi_buy_th': 49,
#     'boll_buffer': 0.0954,
#     'tw_price_th': 4.6432,
#     'tm_price_th': 49.6071,
#     'tw_vol_th': 67.3719,
#     'tm_vol_th': 228.6710,
#     'rsi_sell_th': 10.5987
# }

# # ── 2. 저장 경로 설정 ──
# save_dir = r"C:\Users\LabPC\OneDrive\주식\Results"
# os.makedirs(save_dir, exist_ok=True)

# # ── 3. 데이터 로딩 (파일 경로 수정 요망) ──
# df = pd.read_csv(r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv", encoding='utf-8-sig')
# df['날짜'] = pd.to_datetime(df['날짜'])
# df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')]

# vix = pd.read_csv(r"C:\Users\LabPC\OneDrive\주식\Macro Data\VIX.csv", parse_dates=[0], encoding='utf-8-sig')
# vix.columns = ['날짜', 'VIX']
# df = df.merge(vix, on='날짜', how='left')

# # ── 4. 조건 함수 정의 ──
# def buy_signal(row):
#     return (
#         row['RSI (14일)'] < params['rsi_buy_th'] and
#         row['종가'] < row['볼린저밴드 하단'] * (1 + params['boll_buffer']) and
#         row['MACD'] > row['MACD 시그널'] and
#         row['SMA 5일'] > row['SMA 10일'] and
#         row['SMA 5일'] < row['SMA 60일'] and
#         row['가격 상승률 (2주)'] < params['tw_price_th'] and
#         row['가격 상승률 (3개월)'] < params['tm_price_th'] and
#         row['거래량 상승률 (2주)'] < params['tw_vol_th'] and
#         row['거래량 상승률 (3개월)'] < params['tm_vol_th'] and
#         row['VIX'] >= 60
#     )

# def sell_signal(row):
#     return row['RSI (14일)'] > params['rsi_sell_th']

# # ── 5. 백테스트 함수 ──
# def run_backtest(extra_on_buy=False):
#     cash, shares, total_invested = 10000, 0, 10000
#     logs = []

#     for _, row in df.iterrows():
#         price, date = row['종가'], row['날짜']
#         if buy_signal(row):
#             invest = 10000 if extra_on_buy else (cash if shares == 0 else 0)
#             if invest > 0:
#                 cash += invest if extra_on_buy else 0
#                 total_invested += invest if extra_on_buy else 0
#                 shares += invest / price
#                 cash -= invest
#                 logs.append([date, "BUY", price, shares, cash, shares * price + cash, (shares * price + cash) / total_invested * 100])
#         elif sell_signal(row) and shares > 0:
#             cash += shares * price
#             logs.append([date, "SELL", price, shares, cash, cash, cash / total_invested * 100])
#             shares = 0

#     final_value = cash + shares * df.iloc[-1]['종가']
#     roi = final_value / total_invested * 100
#     return pd.DataFrame(logs, columns=["날짜", "액션", "가격", "보유주", "현금", "총자산", "ROI(%)"]), roi

# # ── 6. 실행 및 CSV 저장 ──
# df1, roi1 = run_backtest(extra_on_buy=False)
# df2, roi2 = run_backtest(extra_on_buy=True)

# df1.to_csv(os.path.join(save_dir, "one_time_investment.csv"), index=False, encoding='utf-8-sig')
# df2.to_csv(os.path.join(save_dir, "extra_10000_every_buy.csv"), index=False, encoding='utf-8-sig')

# print(f"✅ 1회 투자 ROI: {roi1:.2f}% → 저장 위치: {save_dir}\\one_time_investment.csv")
# print(f"✅ 매수마다 1만 추가 투자 ROI: {roi2:.2f}% → 저장 위치: {save_dir}\\extra_10000_every_buy.csv")


In [ ]:
import os
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────
# 1) 최종 파라미터 입력 (▶ [Final Combined] 결과)
# ─────────────────────────────────────────────────────
params = {
    'vix_buy_th':   37.28994809694973,
    'vix_sell_th':   5.799125806917302,
    'rsi_buy_th':   78.49990960271758,
    'boll_buffer':   0.09570817361851704,
    'tw_price_th':  17.28209132296718,
    'tm_price_th':  11.203875933889309,
    'tw_vol_th':    14.576505184329823,
    'tm_vol_th':     8.742939261779691,
    'rsi_sell_th':  89.97075788097432
}
# ▶ [Final Combined] Best ROI: 5688.26%

# ─────────────────────────────────────────────────────
# 2) 경로 설정
# ─────────────────────────────────────────────────────
input_csv    = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder = r"C:\Users\seung\OneDrive\주식\Macro Data"
save_dir     = r"C:\Users\seung\OneDrive\주식\Results"
os.makedirs(save_dir, exist_ok=True)

# ─────────────────────────────────────────────────────
# 3) 데이터 로딩 및 병합
# ─────────────────────────────────────────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2020-01-02') & (df['날짜'] < '2025-01-01')].reset_index(drop=True)

vix = pd.read_csv(os.path.join(macro_folder, "VIX.csv"),
                  parse_dates=[0], encoding='utf-8-sig')
vix.columns = ['날짜','VIX']
df = df.merge(vix, on='날짜', how='left')

# ─────────────────────────────────────────────────────
# 4) 시그널 정의
# ─────────────────────────────────────────────────────
def is_macro_buy(row):
    return not np.isnan(row['VIX']) and row['VIX'] >= params['vix_buy_th']

def is_macro_sell(row):
    return not np.isnan(row['VIX']) and row['VIX'] <= params['vix_sell_th']

def is_rule_buy(row):
    return (
        row['RSI (14일)']           < params['rsi_buy_th'] and
        row['종가']                 < row['볼린저밴드 하단'] * (1 + params['boll_buffer']) and
        row['MACD']                 > row['MACD 시그널'] and
        row['SMA 5일']              > row['SMA 10일'] and
        row['SMA 5일']              < row['SMA 60일'] and
        row['가격 상승률 (2주)']     < params['tw_price_th'] and
        row['가격 상승률 (3개월)']   < params['tm_price_th'] and
        row['거래량 상승률 (2주)']   < params['tw_vol_th'] and
        row['거래량 상승률 (3개월)'] < params['tm_vol_th']
    )

def is_rule_sell(row):
    return row['RSI (14일)'] > params['rsi_sell_th']

# ─────────────────────────────────────────────────────
# 5) 백테스트 함수
# ─────────────────────────────────────────────────────
def run_backtest(extra_on_buy=False):
    cash = 10_000.0
    shares = 0.0
    total_injected = 10_000.0  # 최초 투입
    logs = []

    for _, row in df.iterrows():
        date, price = row['날짜'], row['종가']

        # 1) Macro 강제 매수
        if shares == 0 and is_macro_buy(row):
            shares = cash / price
            cash = 0.0
            logs.append([
                date, 'BUY_MACRO_VIX', price, shares,
                cash, shares*price, round((shares*price)/total_injected*100,2)
            ])
            continue

        # 2) Macro 강제 매도
        if shares > 0 and is_macro_sell(row):
            proceeds = shares * price
            cash += proceeds
            logs.append([
                date, 'SELL_MACRO_VIX', price, shares,
                cash, cash, round(cash/total_injected*100,2)
            ])
            shares = 0.0
            continue

        # 3) Rule 기반 매수
        if shares == 0 and is_rule_buy(row):
            invest = 10_000.0 if extra_on_buy else cash
            if invest > 0:
                # 추가 투입 여부
                if extra_on_buy:
                    cash += invest
                    total_injected += invest
                qty = invest / price
                shares += qty
                cash -= invest
                asset = cash + shares * price
                logs.append([
                    date, 'BUY_RULE', price, qty,
                    cash, asset, round(asset/total_injected*100,2)
                ])

        # 4) Rule 기반 매도
        elif shares > 0 and is_rule_sell(row):
            proceeds = shares * price
            cash += proceeds
            asset = cash
            logs.append([
                date, 'SELL_RULE', price, shares,
                cash, asset, round(asset/total_injected*100,2)
            ])
            shares = 0.0

    # 5) 최종 청산
    if shares > 0:
        date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
        proceeds = shares * price
        cash += proceeds
        asset = cash
        logs.append([
            date, 'LIQUIDATE', price, shares,
            cash, asset, round(asset/total_injected*100,2)
        ])

    cols = ["날짜","액션","가격","보유주","현금","총자산","ROI(%)"]
    return pd.DataFrame(logs, columns=cols), round((cash)/total_injected*100,2)

# ─────────────────────────────────────────────────────
# 6) 시뮬레이션 & CSV 저장
# ─────────────────────────────────────────────────────
df_once, roi_once = run_backtest(extra_on_buy=False)
df_extra, roi_extra = run_backtest(extra_on_buy=True)

csv1 = os.path.join(save_dir, "one_time_investment.csv")
csv2 = os.path.join(save_dir, "extra_10000_every_buy.csv")
df_once.to_csv(csv1, index=False, encoding='utf-8-sig')
df_extra.to_csv(csv2, index=False, encoding='utf-8-sig')

print(f"✅ 1회 투자 ROI: {roi_once:.2f}% → {csv1}")
print(f"✅ 매수마다 10,000 추가 투자 ROI: {roi_extra:.2f}% → {csv2}")


In [ ]:
▶ [TPE] Best ROI: 5588.42
▶ [TPE] Best Parameters:
    • vix_buy_th: 34.78529194184235
    • vix_sell_th: 5.799125806917302
    • rsi_buy_th: 78.49990960271758
    • boll_buffer: 0.09570817361851704
    • tw_price_th: 17.28209132296718
    • tm_price_th: 11.203875933889309
    • tw_vol_th: 14.576505184329823
    • tm_vol_th: 8.742939261779691
    • rsi_sell_th: 89.82169909044313

In [ ]:
#마지막


In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances

# ─────────────────────────────────────────────────────────────
# 1) 경로 설정
# ─────────────────────────────────────────────────────────────
input_csv      = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\seung\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\seung\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

# ─────────────────────────────────────────────────────────────
# 2) 데이터 로딩 및 전처리
# ─────────────────────────────────────────────────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2019-01-02') & (df['날짜'] < '2025-06-24')].reset_index(drop=True)

vix = pd.read_csv(os.path.join(macro_folder, "VIX.csv"),
                  parse_dates=[0], encoding='utf-8-sig')
vix.columns = ['날짜','VIX']
df = df.merge(vix, on='날짜', how='left')

# ─────────────────────────────────────────────────────────────
# 3) 전략용 신호 함수
# ─────────────────────────────────────────────────────────────
def is_macro_buy(row, p):
    return not np.isnan(row['VIX']) and row['VIX'] >= p['vix_buy_th']

def is_macro_sell(row, p):
    return not np.isnan(row['VIX']) and row['VIX'] <= p['vix_sell_th']

def is_rule_buy(row, p):
    return (
        row['RSI (14일)']           < p['rsi_buy_th'] and
        row['종가']                 < row['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
        row['MACD']                 > row['MACD 시그널'] and
        row['SMA 5일']              > row['SMA 10일'] and
        row['SMA 5일']              < row['SMA 60일'] and
        row['가격 상승률 (2주)']     < p['tw_price_th'] and
        row['가격 상승률 (3개월)']   < p['tm_price_th'] and
        row['거래량 상승률 (2주)']   < p['tw_vol_th'] and
        row['거래량 상승률 (3개월)'] < p['tm_vol_th']
    )

def is_rule_sell(row, p):
    return row['RSI (14일)'] > p['rsi_sell_th']

# ─────────────────────────────────────────────────────────────
# 4) ROI 계산용 백테스트 (현금 전액 투자, 추가 자금 투입 없음)
# ─────────────────────────────────────────────────────────────
def backtest_roi(p):
    cash, shares = 10_000.0, 0.0
    for _, row in df.iterrows():
        price, vix = row['종가'], row['VIX']
        # 1) Macro Buy
        if shares==0 and is_macro_buy(row,p):
            shares = cash/price; cash=0.0; continue
        # 2) Macro Sell
        if shares>0 and is_macro_sell(row,p):
            cash = shares*price; shares=0.0; continue
        # 3) Rule Buy
        if shares==0 and is_rule_buy(row,p):
            shares = cash/price; cash=0.0
        # 4) Rule Sell
        elif shares>0 and is_rule_sell(row,p):
            cash = shares*price; shares=0.0
    final = cash + shares*df.iloc[-1]['종가']
    return (final - 10_000.0)/10_000.0*100

# ─────────────────────────────────────────────────────────────
# 5) 1단계: TPE Sampler 로 빠르게 탐색
# ─────────────────────────────────────────────────────────────
def obj_tpe(trial):
    vb = trial.suggest_float("vix_buy_th",  0,100)
    vs = trial.suggest_float("vix_sell_th", 0,vb)
    return backtest_roi({
        'vix_buy_th': vb, 'vix_sell_th': vs,
        'rsi_buy_th':   trial.suggest_float("rsi_buy_th", 0,100),
        'boll_buffer':  trial.suggest_float("boll_buffer",0,0.1),
        'tw_price_th':  trial.suggest_float("tw_price_th",0,20),
        'tm_price_th':  trial.suggest_float("tm_price_th",0,50),
        'tw_vol_th':    trial.suggest_float("tw_vol_th",  0,100),
        'tm_vol_th':    trial.suggest_float("tm_vol_th",  0,300),
        'rsi_sell_th':  trial.suggest_float("rsi_sell_th",0,100)
    })

tpe_study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler())
tpe_study.optimize(obj_tpe, n_trials=1000)
best_tpe = tpe_study.best_params
print("▶ [TPE] Best ROI:", round(tpe_study.best_value,2))
print("▶ [TPE] Params:")
for k,v in best_tpe.items(): print(f"   • {k}: {v}")

# ─────────────────────────────────────────────────────────────
# 6) 중요도 계산 → 상위 5% 이상만 CMA-ES 2단계에 사용
# ─────────────────────────────────────────────────────────────
imp = get_param_importances(tpe_study)
thr = 0.05
important = [k for k,v in imp.items() if v>=thr]
print("\n▶ [TPE] Importances:")
for k,v in imp.items(): print(f"   • {k}: {v:.3f}")
print("▶ Keep ≥",thr,":", important)

# ─────────────────────────────────────────────────────────────
# 7) 2단계: CMA-ES 로 정밀 탐색
# ─────────────────────────────────────────────────────────────
bounds = {
  "vix_buy_th":(0,100),"vix_sell_th":(0,100),
  "rsi_buy_th":(0,100),"boll_buffer":(0,0.1),
  "tw_price_th":(0,20),"tm_price_th":(0,50),
  "tw_vol_th":(0,100),"tm_vol_th":(0,300),
  "rsi_sell_th":(0,100)
}
# narrow ranges ±20%
narrow = {}
for k in important:
    lo,hi = bounds[k]; bp = best_tpe[k]
    d = 0.2*(hi-lo)
    narrow[k] = (max(lo,bp-d), min(hi,bp+d))

def obj_cma(trial):
    p = {}
    for k,(lo,hi) in bounds.items():
        if k in important:
            lo2,hi2 = narrow[k]
            if k=="vix_sell_th": hi2 = min(hi2, best_tpe["vix_buy_th"])
            p[k] = trial.suggest_float(k, lo2, hi2)
        else:
            p[k] = best_tpe[k]
    return backtest_roi(p)

cma_study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.CmaEsSampler())
cma_study.optimize(obj_cma, n_trials=1000)
best_cma = cma_study.best_params
print("\n▶ [CMA-ES] Best ROI:", round(cma_study.best_value,2))
print("▶ [CMA-ES] Params:")
for k,v in best_cma.items(): print(f"   • {k}: {v}")

# ─────────────────────────────────────────────────────────────
# 8) 최종 조합 생성 & 출력
# ─────────────────────────────────────────────────────────────
full_best = best_tpe.copy(); full_best.update(best_cma)
print("\n▶ [Final Combined] Best ROI:", round(cma_study.best_value,2))
print("▶ [Final Combined] All Params:")
for k,v in full_best.items(): print(f"   • {k}: {v}")

# ─────────────────────────────────────────────────────────────
# 9) 시뮬레이션용 백테스트 (로그 기록 포함)
# ─────────────────────────────────────────────────────────────
def run_backtest(params, extra_on_buy=False):
    cash, shares = 10_000.0, 0.0
    total_injected = 10_000.0
    logs = []
    for _, row in df.iterrows():
        date, price = row['날짜'], row['종가']
        # MACRO BUY
        if shares==0 and is_macro_buy(row,params):
            shares = cash/price; cash=0.0
            logs.append([date,'BUY_MACRO_VIX',price,shares,cash,shares*price])
            continue
        # MACRO SELL
        if shares>0 and is_macro_sell(row,params):
            cash = shares*price; shares=0.0
            logs.append([date,'SELL_MACRO_VIX',price,0,cash,cash])
            continue
        # RULE BUY
        if shares==0 and is_rule_buy(row,params):
            invest = 10_000.0 if extra_on_buy else cash
            if invest>0:
                if extra_on_buy:
                    cash += invest; total_injected += invest
                qty = invest/price
                shares=qty; cash-=invest
                logs.append([date,'BUY_RULE',price,qty,cash,cash+shares*price])
        # RULE SELL
        elif shares>0 and is_rule_sell(row,params):
            cash = shares*price
            logs.append([date,'SELL_RULE',price,0,cash,cash])
            shares=0.0
    # LIQUIDATE
    if shares>0:
        date, price = df.iloc[-1]['날짜'],df.iloc[-1]['종가']
        cash = shares*price; logs.append([date,'LIQUIDATE',price,0,cash,cash])
    cols = ["날짜","액션","가격","보유주","현금","총자산"]
    return pd.DataFrame(logs, columns=cols), round(cash/total_injected*100,2)

# ─────────────────────────────────────────────────────────────
# 10) 시뮬레이션 실행 & CSV 저장
# ─────────────────────────────────────────────────────────────
df_once, roi_once = run_backtest(full_best, extra_on_buy=False)
df_extra, roi_extra = run_backtest(full_best, extra_on_buy=True)

path1 = os.path.join(results_folder,"one_time_investment.csv")
path2 = os.path.join(results_folder,"extra_10000_every_buy.csv")
df_once.to_csv(path1, index=False, encoding='utf-8-sig')
df_extra.to_csv(path2, index=False, encoding='utf-8-sig')

print(f"\n✅ 1회 투자 ROI: {roi_once:.2f}% → {path1}")
print(f"✅ 매수마다 10,000 추가 ROI: {roi_extra:.2f}% → {path2}")


In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances

# ─────────────────────────────────────────────────────────────
# 1) 경로 설정
# ─────────────────────────────────────────────────────────────
# input_csv      = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# macro_folder   = r"C:\Users\seung\OneDrive\주식\Macro Data"
# results_folder = r"C:\Users\seung\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"

# ─────────────────────────────────────────────────────────────
# 2) 데이터 로딩 및 전처리
# ─────────────────────────────────────────────────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2019-01-02') & (df['날짜'] < '2025-06-24')].reset_index(drop=True)

vix = pd.read_csv(os.path.join(macro_folder, "VIX.csv"),
                  parse_dates=[0], encoding='utf-8-sig')
vix.columns = ['날짜','VIX']
df = df.merge(vix, on='날짜', how='left')

# ─────────────────────────────────────────────────────────────
# 3) 전략용 신호 함수
# ─────────────────────────────────────────────────────────────
def is_macro_buy(row, p):
    return not np.isnan(row['VIX']) and row['VIX'] >= p['vix_buy_th']

def is_macro_sell(row, p):
    return not np.isnan(row['VIX']) and row['VIX'] <= p['vix_sell_th']

def is_rule_buy(row, p):
    return (
        row['RSI (14일)']           < p['rsi_buy_th'] and
        row['종가']                 < row['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
        row['MACD']                 > row['MACD 시그널'] and
        row['SMA 5일']              > row['SMA 10일'] and
        row['SMA 5일']              < row['SMA 60일'] and
        row['가격 상승률 (2주)']     < p['tw_price_th'] and
        row['가격 상승률 (3개월)']   < p['tm_price_th'] and
        row['거래량 상승률 (2주)']   < p['tw_vol_th'] and
        row['거래량 상승률 (3개월)'] < p['tm_vol_th']
    )

def is_rule_sell(row, p):
    return row['RSI (14일)'] > p['rsi_sell_th']

# ─────────────────────────────────────────────────────────────
# 4) ROI 계산용 백테스트 (전액 투자, 추가 자금 투입 없음)
# ─────────────────────────────────────────────────────────────
def backtest_roi(p):
    cash, shares = 10_000.0, 0.0
    for _, row in df.iterrows():
        price, vix = row['종가'], row['VIX']
        # 1) Macro Buy
        if shares == 0 and is_macro_buy(row, p):
            shares = cash / price
            cash = 0.0
            continue
        # 2) Macro Sell
        if shares > 0 and is_macro_sell(row, p):
            cash = shares * price
            shares = 0.0
            continue
        # 3) Rule Buy
        if shares == 0 and is_rule_buy(row, p):
            shares = cash / price
            cash = 0.0
        # 4) Rule Sell
        elif shares > 0 and is_rule_sell(row, p):
            cash = shares * price
            shares = 0.0
    final = cash + shares * df.iloc[-1]['종가']
    return (final - 10_000.0) / 10_000.0 * 100

# ─────────────────────────────────────────────────────────────
# 5) 1단계: TPE Sampler 로 빠르게 탐색
# ─────────────────────────────────────────────────────────────
def obj_tpe(trial):
    vb = trial.suggest_float("vix_buy_th",  0, 100)
    vs = trial.suggest_float("vix_sell_th", 0, vb)
    return backtest_roi({
        'vix_buy_th':  vb,
        'vix_sell_th': vs,
        'rsi_buy_th':   trial.suggest_float("rsi_buy_th", 0, 100),
        'boll_buffer':  trial.suggest_float("boll_buffer", 0, 0.1),
        'tw_price_th':  trial.suggest_float("tw_price_th", 0, 20),
        'tm_price_th':  trial.suggest_float("tm_price_th", 0, 50),
        'tw_vol_th':    trial.suggest_float("tw_vol_th", 0, 100),
        'tm_vol_th':    trial.suggest_float("tm_vol_th", 0, 300),
        'rsi_sell_th':  trial.suggest_float("rsi_sell_th", 0, 100),
    })

tpe_study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler())
tpe_study.optimize(obj_tpe, n_trials=1000)
best_tpe = tpe_study.best_params

print("▶ [TPE] Best ROI:", round(tpe_study.best_value, 2))
print("▶ [TPE] Params:")
for k, v in best_tpe.items():
    print(f"   • {k}: {v}")

# ─────────────────────────────────────────────────────────────
# 6) 중요도 계산
# ─────────────────────────────────────────────────────────────
imp = get_param_importances(tpe_study)
thr = 0.05
important = [k for k, v in imp.items() if v >= thr]

print("\n▶ [TPE] Importances:")
for k, v in imp.items():
    print(f"   • {k}: {v:.3f}")
print("▶ Keep ≥", thr, ":", important)

# ─────────────────────────────────────────────────────────────
# 7) 2단계: CMA-ES 로 정밀 탐색 (중요 파라미터만 사용)
# ─────────────────────────────────────────────────────────────
bounds = {
    "vix_buy_th":  (0, 100), "vix_sell_th": (0, 100),
    "rsi_buy_th":  (0, 100), "boll_buffer":  (0, 0.1),
    "tw_price_th": (0, 20),  "tm_price_th":  (0, 50),
    "tw_vol_th":   (0, 100), "tm_vol_th":   (0, 300),
    "rsi_sell_th": (0, 100)
}
narrow = {}
for k in important:
    lo, hi = bounds[k]
    bp = best_tpe[k]
    d = 0.2 * (hi - lo)
    narrow[k] = (max(lo, bp - d), min(hi, bp + d))

def obj_cma(trial):
    p = {}
    for k, (lo, hi) in bounds.items():
        if k in important:
            lo2, hi2 = narrow[k]
            if k == "vix_sell_th":
                hi2 = min(hi2, best_tpe["vix_buy_th"])
            p[k] = trial.suggest_float(k, lo2, hi2)
        else:
            p[k] = best_tpe[k]
    return backtest_roi(p)

cma_study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.CmaEsSampler())
cma_study.optimize(obj_cma, n_trials=1000)
best_cma = cma_study.best_params

print("\n▶ [CMA-ES] Best ROI:", round(cma_study.best_value, 2))
print("▶ [CMA-ES] Params:")
for k, v in best_cma.items():
    print(f"   • {k}: {v}")

# ─────────────────────────────────────────────────────────────
# 8) 최종 조합 생성 & 출력
# ─────────────────────────────────────────────────────────────
full_best = best_tpe.copy()
full_best.update(best_cma)

print("\n▶ [Final Combined] Best ROI:", round(cma_study.best_value, 2))
print("▶ [Final Combined] All Params:")
for k, v in full_best.items():
    print(f"   • {k}: {v}")

# ─────────────────────────────────────────────────────────────
# 9) 시뮬레이션용 백테스트 (로그 기록 포함)
# ─────────────────────────────────────────────────────────────
def run_backtest(params, extra_on_buy=False):
    cash, shares = 10_000.0, 0.0
    total_injected = 10_000.0
    logs = []

    for _, row in df.iterrows():
        date, price, vix = row['날짜'], row['종가'], row['VIX']

        # 1) Macro 강제 매수
        if shares == 0 and is_macro_buy(row, params):
            shares = cash / price
            cash = 0.0
            action = f"BUY_MACRO_VIX (VIX={vix:.2f}≥{params['vix_buy_th']:.2f})"
            logs.append([date, action, price, shares, cash, shares*price])
            continue

        # 2) Macro 강제 매도
        if shares > 0 and is_macro_sell(row, params):
            proceeds = shares * price
            cash += proceeds
            action = f"SELL_MACRO_VIX (VIX={vix:.2f}≤{params['vix_sell_th']:.2f})"
            logs.append([date, action, price, 0.0, cash, cash])
            shares = 0.0
            continue

        # 3) Rule 기반 매수
        if shares == 0 and is_rule_buy(row, params):
            invest = 10_000.0 if extra_on_buy else cash
            if invest > 0:
                if extra_on_buy:
                    cash += invest
                    total_injected += invest
                qty = invest / price
                shares = qty
                cash -= invest
                action = "BUY_RULE (all technical conditions)"
                logs.append([date, action, price, qty, cash, cash + shares*price])

        # 4) Rule 기반 매도
        elif shares > 0 and is_rule_sell(row, params):
            proceeds = shares * price
            cash += proceeds
            action = f"SELL_RULE (RSI={row['RSI (14일)']:.2f}≥{params['rsi_sell_th']:.2f})"
            logs.append([date, action, price, 0.0, cash, cash])
            shares = 0.0

    # 5) 최종 청산
    if shares > 0:
        date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
        proceeds = shares * price
        cash += proceeds
        action = "LIQUIDATE (END)"
        logs.append([date, action, price, 0.0, cash, cash])

    cols = ["날짜","액션","가격","보유주","현금","총자산"]
    return pd.DataFrame(logs, columns=cols), round(cash/total_injected*100, 2)

# ─────────────────────────────────────────────────────────────
# 10) 시뮬레이션 실행 & CSV 저장
# ─────────────────────────────────────────────────────────────
df_once, roi_once = run_backtest(full_best, extra_on_buy=False)
df_extra, roi_extra = run_backtest(full_best, extra_on_buy=True)

csv1 = os.path.join(results_folder, "one_time_investment.csv")
csv2 = os.path.join(results_folder, "extra_10000_every_buy.csv")
df_once.to_csv(csv1, index=False, encoding='utf-8-sig')
df_extra.to_csv(csv2, index=False, encoding='utf-8-sig')

print(f"\n✅ 1회 투자 ROI: {roi_once:.2f}% → {csv1}")
print(f"✅ 매수마다 10,000 추가 투자 ROI: {roi_extra:.2f}% → {csv2}")


In [ ]:
import os
import pandas as pd
import numpy as np
import optuna
from optuna.importance import get_param_importances

# ─────────────────────────────────────────────────────────────
# 1) 경로 설정
# ─────────────────────────────────────────────────────────────
# input_csv      = r"C:\Users\seung\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
# macro_folder   = r"C:\Users\seung\OneDrive\주식\Macro Data"
# results_folder = r"C:\Users\seung\OneDrive\주식\Results"
os.makedirs(results_folder, exist_ok=True)

input_csv      = r"C:\Users\LabPC\OneDrive\주식\Processed Data\Tesla Stock Price History_지표포함.csv"
macro_folder   = r"C:\Users\LabPC\OneDrive\주식\Macro Data"
results_folder = r"C:\Users\LabPC\OneDrive\주식\Results"

# ─────────────────────────────────────────────────────────────
# 2) 데이터 로딩 및 전처리
# ─────────────────────────────────────────────────────────────
df = pd.read_csv(input_csv, encoding='utf-8-sig')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df[(df['날짜'] >= '2019-01-02') & (df['날짜'] < '2025-06-24')].reset_index(drop=True)

vix = pd.read_csv(os.path.join(macro_folder, "VIX.csv"),
                  parse_dates=[0], encoding='utf-8-sig')
vix.columns = ['날짜','VIX']
df = df.merge(vix, on='날짜', how='left')

# ─────────────────────────────────────────────────────────────
# 3) 전략용 신호 함수
# ─────────────────────────────────────────────────────────────
def is_macro_buy(row, p):
    return not np.isnan(row['VIX']) and row['VIX'] >= p['vix_buy_th']

def is_macro_sell(row, p):
    return not np.isnan(row['VIX']) and row['VIX'] <= p['vix_sell_th']

def is_rule_buy(row, p):
    return (
        row['RSI (14일)']           < p['rsi_buy_th'] and
        row['종가']                 < row['볼린저밴드 하단'] * (1 + p['boll_buffer']) and
        row['MACD']                 > row['MACD 시그널'] and
        row['SMA 5일']              > row['SMA 10일'] and
        row['SMA 5일']              < row['SMA 60일'] and
        row['가격 상승률 (2주)']     < p['tw_price_th'] and
        row['가격 상승률 (3개월)']   < p['tm_price_th'] and
        row['거래량 상승률 (2주)']   < p['tw_vol_th'] and
        row['거래량 상승률 (3개월)'] < p['tm_vol_th']
    )

def is_rule_sell(row, p):
    return row['RSI (14일)'] > p['rsi_sell_th']

# ─────────────────────────────────────────────────────────────
# 4) ROI 계산용 백테스트 (전액 투자, 추가 자금 투입 없음)
# ─────────────────────────────────────────────────────────────
def backtest_roi(p):
    cash, shares = 10_000.0, 0.0
    for _, row in df.iterrows():
        price, vix = row['종가'], row['VIX']
        # 1) Macro Buy
        if shares == 0 and is_macro_buy(row, p):
            shares = cash / price
            cash = 0.0
            continue
        # 2) Macro Sell
        if shares > 0 and is_macro_sell(row, p):
            cash = shares * price
            shares = 0.0
            continue
        # 3) Rule Buy
        if shares == 0 and is_rule_buy(row, p):
            shares = cash / price
            cash = 0.0
        # 4) Rule Sell
        elif shares > 0 and is_rule_sell(row, p):
            cash = shares * price
            shares = 0.0
    final = cash + shares * df.iloc[-1]['종가']
    return (final - 10_000.0) / 10_000.0 * 100

# ─────────────────────────────────────────────────────────────
# 5) 1단계: TPE Sampler 로 빠르게 탐색
# ─────────────────────────────────────────────────────────────
def obj_tpe(trial):
    vb = trial.suggest_float("vix_buy_th",  0, 100)
    vs = trial.suggest_float("vix_sell_th", 0, vb)
    return backtest_roi({
        'vix_buy_th':  vb,
        'vix_sell_th': vs,
        'rsi_buy_th':   trial.suggest_float("rsi_buy_th", 0, 100),
        'boll_buffer':  trial.suggest_float("boll_buffer", 0, 0.1),
        'tw_price_th':  trial.suggest_float("tw_price_th", 0, 20),
        'tm_price_th':  trial.suggest_float("tm_price_th", 0, 50),
        'tw_vol_th':    trial.suggest_float("tw_vol_th", 0, 100),
        'tm_vol_th':    trial.suggest_float("tm_vol_th", 0, 300),
        'rsi_sell_th':  trial.suggest_float("rsi_sell_th", 0, 100),
    })

tpe_study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler())
tpe_study.optimize(obj_tpe, n_trials=1000)
best_tpe = tpe_study.best_params

print("▶ [TPE] Best ROI:", round(tpe_study.best_value, 2))
print("▶ [TPE] Params:")
for k, v in best_tpe.items():
    print(f"   • {k}: {v}")

# ─────────────────────────────────────────────────────────────
# 6) 중요도 계산
# ─────────────────────────────────────────────────────────────
imp = get_param_importances(tpe_study)
thr = 0.05
important = [k for k, v in imp.items() if v >= thr]

print("\n▶ [TPE] Importances:")
for k, v in imp.items():
    print(f"   • {k}: {v:.3f}")
print("▶ Keep ≥", thr, ":", important)

# ─────────────────────────────────────────────────────────────
# 7) 2단계: CMA-ES 로 정밀 탐색 (중요 파라미터만 사용)
# ─────────────────────────────────────────────────────────────
bounds = {
    "vix_buy_th":  (0, 100), "vix_sell_th": (0, 100),
    "rsi_buy_th":  (0, 100), "boll_buffer":  (0, 0.1),
    "tw_price_th": (0, 20),  "tm_price_th":  (0, 50),
    "tw_vol_th":   (0, 100), "tm_vol_th":   (0, 300),
    "rsi_sell_th": (0, 100)
}
narrow = {}
for k in important:
    lo, hi = bounds[k]
    bp = best_tpe[k]
    d = 0.2 * (hi - lo)
    narrow[k] = (max(lo, bp - d), min(hi, bp + d))

def obj_cma(trial):
    p = {}
    for k, (lo, hi) in bounds.items():
        if k in important:
            lo2, hi2 = narrow[k]
            if k == "vix_sell_th":
                hi2 = min(hi2, best_tpe["vix_buy_th"])
            p[k] = trial.suggest_float(k, lo2, hi2)
        else:
            p[k] = best_tpe[k]
    return backtest_roi(p)

cma_study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.CmaEsSampler())
cma_study.optimize(obj_cma, n_trials=1000)
best_cma = cma_study.best_params

print("\n▶ [CMA-ES] Best ROI:", round(cma_study.best_value, 2))
print("▶ [CMA-ES] Params:")
for k, v in best_cma.items():
    print(f"   • {k}: {v}")

# ─────────────────────────────────────────────────────────────
# 8) 최종 조합 생성 & 출력
# ─────────────────────────────────────────────────────────────
full_best = best_tpe.copy()
full_best.update(best_cma)

print("\n▶ [Final Combined] Best ROI:", round(cma_study.best_value, 2))
print("▶ [Final Combined] All Params:")
for k, v in full_best.items():
    print(f"   • {k}: {v}")

# ─────────────────────────────────────────────────────────────
# 9) 시뮬레이션용 백테스트 (연속 매수 + 쿨다운 + 실시간 ROI/포맷)
# ─────────────────────────────────────────────────────────────
def run_backtest(params, extra_on_buy=False, cooldown_days=0):
    cash, shares = 10_000.0, 0.0
    total_injected = 10_000.0
    logs = []
    last_buy_date = None  # 최근 매수 날짜 기록

    for _, row in df.iterrows():
        date, price, vix = row['날짜'], row['종가'], row['VIX']

        # 1) Macro 매수 (쿨다운 체크)
        if is_macro_buy(row, params):
            if last_buy_date is None or (date - last_buy_date).days >= cooldown_days:
                invest = 10_000.0 if extra_on_buy else cash
                if invest > 0:
                    if extra_on_buy:
                        cash += invest
                        total_injected += invest
                    qty = invest / price
                    shares += qty
                    cash -= invest
                    last_buy_date = date
                    action = f"BUY_MACRO_VIX (VIX={vix:.2f}≥{params['vix_buy_th']:.2f})"
                    total_asset = cash + shares * price
                    roi_pct     = total_asset / total_injected * 100
                    logs.append([
                        date, action, price, shares,
                        cash, total_asset, round(roi_pct, 2)
                    ])
                continue

        # 2) Macro 매도
        if shares > 0 and is_macro_sell(row, params):
            proceeds = shares * price
            cash += proceeds
            action = f"SELL_MACRO_VIX (VIX={vix:.2f}≤{params['vix_sell_th']:.2f})"
            total_asset = cash
            roi_pct     = total_asset / total_injected * 100
            logs.append([
                date, action, price, 0.0,
                cash, total_asset, round(roi_pct, 2)
            ])
            shares = 0.0
            last_buy_date = None
            continue

        # 3) Rule 기반 연속 매수 (쿨다운 체크)
        if is_rule_buy(row, params):
            if last_buy_date is None or (date - last_buy_date).days >= cooldown_days:
                invest = 10_000.0 if extra_on_buy else cash
                if invest > 0:
                    if extra_on_buy:
                        cash += invest
                        total_injected += invest
                    qty = invest / price
                    shares += qty
                    cash -= invest
                    last_buy_date = date
                    action = "BUY_RULE (all technical conditions)"
                    total_asset = cash + shares * price
                    roi_pct     = total_asset / total_injected * 100
                    logs.append([
                        date, action, price, shares,
                        cash, total_asset, round(roi_pct, 2)
                    ])

        # 4) Rule 기반 매도
        elif shares > 0 and is_rule_sell(row, params):
            proceeds = shares * price
            cash += proceeds
            action = f"SELL_RULE (RSI={row['RSI (14일)']:.2f}≥{params['rsi_sell_th']:.2f})"
            total_asset = cash
            roi_pct     = total_asset / total_injected * 100
            logs.append([
                date, action, price, 0.0,
                cash, total_asset, round(roi_pct, 2)
            ])
            shares = 0.0
            last_buy_date = None

    # 5) 최종 청산
    if shares > 0:
        date, price = df.iloc[-1]['날짜'], df.iloc[-1]['종가']
        proceeds = shares * price
        cash += proceeds
        action = "LIQUIDATE (END)"
        total_asset = cash
        roi_pct     = total_asset / total_injected * 100
        logs.append([
            date, action, price, 0.0,
            cash, total_asset, round(roi_pct, 2)
        ])

    # DataFrame 생성 & 포맷 적용
    cols = ["날짜","액션","가격","보유주","현금","총자산","ROI(%)"]
    df_logs = pd.DataFrame(logs, columns=cols)

    # 천 단위 콤마 포맷 (소수점 없이) + ROI 뒤에 '%' 추가
    df_logs["현금"]    = df_logs["현금"].map(lambda x: f"{x:,.0f}")
    df_logs["총자산"]  = df_logs["총자산"].map(lambda x: f"{x:,.0f}")
    df_logs["ROI(%)"]  = df_logs["ROI(%)"].map(lambda x: f"{x:.2f}%")

    return df_logs, round(cash/total_injected*100, 2)


# ─────────────────────────────────────────────────────────────
# 10) 시뮬레이션 실행 & CSV 저장
# ─────────────────────────────────────────────────────────────
df_once, roi_once = run_backtest(full_best, extra_on_buy=False, cooldown_days=0)
df_extra, roi_extra = run_backtest(full_best, extra_on_buy=True,  cooldown_days=0)

csv1 = os.path.join(results_folder, "one_time_investment.csv")
csv2 = os.path.join(results_folder, "extra_10000_every_buy.csv")
df_once.to_csv(csv1, index=False, encoding='utf-8-sig')
df_extra.to_csv(csv2, index=False, encoding='utf-8-sig')

print(f"\n✅ 1회 투자 최종 ROI: {roi_once:.2f}% → {csv1}")
print(f"✅ 매수마다 10,000 추가 투자 최종 ROI: {roi_extra:.2f}% → {csv2}")
